# Chapter 17: Network

> *"We set up a telephone connection between us and the guys at SRI...
> We typed the L and we asked on the phone, 'Do you see the L?' 'Yes, we
> see the L.' We typed the O, and we asked, 'Do you see the O?' 'Yes, we
> see the O.' Then we typed the G, and the system crashed."*  
> — **Leonard Kleinrock**, on the first ARPANET transmission (29 October
> 1969)

------------------------------------------------------------------------

### 17.1 The Physics of Separation and The Fallacy of Distributed Systems

In all our computational models so far, we have taken a fundamental
assumption for granted: **instantaneous, deterministic access to
memory**. When your CPU fetches a 64-bit word from the L1 cache or
evaluates a pointer into heap memory, the transaction completes within
sub-nanosecond to tens-of-nanosecond scales. Memory buses are governed
by tight, synchronous physical clocks. A pointer dereference either
succeeds or halts the process via a hardware segmentation fault.

The moment two computing nodes communicate over a physical wire,
fibre-optic cable, or radio frequency channel, this deterministic mental
model collapses (as outlined in Figure 17–1).

<figure id="fig:local-remote">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjlwdCIgaGVpZ2h0PSIyMjlwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjkuMDAgMjI5LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIy
NSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gbG9jYWwgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+bG9jYWw8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBwb2ludHM9IjM2MCwtMjIxIDAsLTIyMSAwLC0xNDkgMzYwLC0xNDkgMzYwLC0yMjEi
Lz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItMjA1IiBmb250LWZhbWls
eT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIx
MC4wMCI+TG9jYWwgTWFjaGluZSBDb250ZXh0PC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3Rh
cnQiIHg9IjUuNSIgeT0iLTE4MiIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2Fu
cyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiYjNDU7IFN5bmNocm9ub3VzIG1l
bW9yeSBidXMgKH4xMCYjNDU7MTAwIG5zIGxhdGVuY3kpPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hv
cj0ic3RhcnQiIHg9IjUuNSIgeT0iLTE3MSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWph
VnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiYjNDU7IFNoYXJlZCBw
aHlzaWNhbCBjbG9jayBhbmQgdW5pZm9ybSBhZGRyZXNzIHNwYWNlPC90ZXh0Pgo8dGV4dCB0ZXh0
LWFuY2hvcj0ic3RhcnQiIHg9IjUuNSIgeT0iLTE2MCIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3
LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiYjNDU7IERl
dGVybWluaXN0aWMgY3Jhc2ggc2VtYW50aWNzIChQcm9jZXNzIGhhbHRzIGNvbXBsZXRlbHkpPC90
ZXh0Pgo8L2c+CjwhLS0gcmVtb3RlIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRp
dGxlPnJlbW90ZTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBv
aW50cz0iMzYwLC03MiAwLC03MiAwLDAgMzYwLDAgMzYwLC03MiIvPgo8dGV4dCB0ZXh0LWFuY2hv
cj0ibWlkZGxlIiB4PSIxODAiIHk9Ii01NiIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWph
VnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlJlbW90ZSBNYWNoaW5l
IENvbnRleHQ8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iMi41IiB5PSItMzMi
IGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBm
b250LXNpemU9IjEwLjAwIj4mIzQ1OyBBc3luY2hyb25vdXMgZXhlY3V0aW9uICh+MSYjNDU7NTAw
IG1zIGxhdGVuY3k6IDEwXjZ4IHNsb3dlciEpPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3Rh
cnQiIHg9IjIuNSIgeT0iLTIyIiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5z
IE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+JiM0NTsgSW5kZXBlbmRlbnQgbWVt
b3J5IHNwYWNlLCBpbmRlcGVuZGVudCBDUFUgYXJjaGl0ZWN0dXJlPC90ZXh0Pgo8dGV4dCB0ZXh0
LWFuY2hvcj0ic3RhcnQiIHg9IjIuNSIgeT0iLTExIiBmb250LWZhbWlseT0iQ291cmllciBOZXcs
IERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+JiM0NTsgUGFy
dGlhbCBmYWlsdXJlIHN0YXRlcyAoTm9kZSBsaXZlcywgYnV0IGxpbmsgaXMgZGVhZCk8L3RleHQ+
CjwvZz4KPCEtLSBsb2NhbCYjNDU7Jmd0O3JlbW90ZSAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0i
ZWRnZSI+Cjx0aXRsZT5sb2NhbCYjNDU7Jmd0O3JlbW90ZTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5v
bmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xODAsLTEzOC42M0MxODAsLTEyMC43NSAxODAsLTEwMC4y
NSAxODAsLTgyLjM2Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2lu
dHM9IjE3Ni41LC0xMzguNzkgMTgwLC0xNDguNzkgMTgzLjUsLTEzOC43OSAxNzYuNSwtMTM4Ljc5
Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE4My41LC04
Mi4xOCAxODAsLTcyLjE4IDE3Ni41LC04Mi4xOCAxODMuNSwtODIuMTgiLz4KPHRleHQgdGV4dC1h
bmNob3I9InN0YXJ0IiB4PSIxODAiIHk9Ii0xMTkiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywg
RGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4gJiMxNjA7UGh5
c2ljYWwgVHJhbnNtaXNzaW9uIE1lZGl1bTwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0
IiB4PSIxODAiIHk9Ii0xMDgiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMg
TW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4gJiMxNjA7KERpc3RhbmNlLCBBdHRl
bnVhdGlvbiw8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iMTgwIiB5PSItOTci
IGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBm
b250LXNpemU9IjEwLjAwIj4gJiMxNjA7UGFja2V0IERyb3BzLCBKaXR0ZXIpPC90ZXh0Pgo8L2c+
CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 17–1: Local and Remote Machine Context</figcaption>
</figure>

When communicating across networks, your software enters the domain of
**Distributed Systems**. In 1994, Peter Deutsch and James Gosling
articulated the *Fallacies of Distributed Computing*—false assumptions
that developers instinctively make when transitioning from
single-machine code to networked architectures:

1.  The network is reliable.
2.  Latency is zero.
3.  Bandwidth is infinite.
4.  The network is secure.
5.  Topology does not change.
6.  There is one administrator.
7.  Transport cost is zero.
8.  The network is homogeneous.

To build software that does not collapse under real-world conditions, we
must understand how raw electrical and optical pulses are structured
into structured messages, how unprincipled byte streams are framed into
discrete data packets, and how to write low-level systems code in Modern
C++ that handles partial failures, out-of-order delivery, and concurrent
I/O.

### 17.2 The Layered Architecture: OSI and TCP/IP

To prevent application code from having to manipulate raw radio
modulation frequencies or Ethernet voltage levels, networks are
structured into **hierarchical protocol layers**. Each layer provides an
abstract service to the layer directly above it, while wrapping
(encapsulating) the data payload from that layer in its own structural
headers (as shown in Figure 17–2).

<figure id="fig:protocol-layers">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDAgOTIwIDM2MCIgd2lkdGg9IjEwMCUiIGhlaWdodD0iMzYwIiB4bWxu
cz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciPgogIDxkZWZzPgogICAgPHN0eWxlPgogICAg
ICA6cm9vdCB7CiAgICAgICAgLS1zdmctYmc6IHZhcigtLWJnLCAjZmZmZmZmKTsKICAgICAgICAt
LXN2Zy1mZzogdmFyKC0tZmcsICMwMDAwMDApOwogICAgICAgIC0tc3ZnLXRpbnQtMS1mYWxsYmFj
azogI2NjY2NjYzsgLyogMjAlIGJsYWNrICsgODAlIHdoaXRlICovCiAgICAgICAgLS1zdmctdGlu
dC0yLWZhbGxiYWNrOiAjOTk5OTk5OyAvKiA0MCUgYmxhY2sgKyA2MCUgd2hpdGUgKi8KICAgICAg
ICAtLXN2Zy10aW50LTE6IGNvbG9yLW1peChpbiBzcmdiLCB2YXIoLS1zdmctZmcpIDIwJSwgdmFy
KC0tc3ZnLWJnKSA4MCUpOwogICAgICAgIC0tc3ZnLXRpbnQtMjogY29sb3ItbWl4KGluIHNyZ2Is
IHZhcigtLXN2Zy1mZykgNDAlLCB2YXIoLS1zdmctYmcpIDYwJSk7CiAgICAgIH0KICAgICAgdGV4
dCAgIHsgZm9udC1mYW1pbHk6ICJBcmlhbCBOYXJyb3ciLCBGcmVlU2Fucywgc2Fucy1zZXJpZjsg
fQogICAgPC9zdHlsZT4KICAgIDxmaWx0ZXIgaWQ9Im5ldFNoYWRvdyIgeD0iMCUiIHk9Ii01JSIg
d2lkdGg9IjExMCUiIGhlaWdodD0iMTIwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9IjAiIGR5
PSI0IiBzdGREZXZpYXRpb249IjQiIGZsb29kLWNvbG9yPSIjMDAwIiBmbG9vZC1vcGFjaXR5PSIw
LjUiLz4KICAgIDwvZmlsdGVyPgogIDwvZGVmcz4KICA8IS0tIExlZnQ6IFRoZSBUQ1AvSVAgQ29u
Y2VwdHVhbCBMYXllcnMgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoMzAsIDQwKSI+CiAg
ICA8dGV4dCB4PSIxNDAiIHk9IjAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTYi
IGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5UQ1AvSVAgQXJjaGl0ZWN0
dXJlPC90ZXh0PgogICAgPCEtLSBBcHBsaWNhdGlvbiBMYXllciAtLT4KICAgIDxyZWN0IHg9IjAi
IHk9IjIwIiB3aWR0aD0iMjgwIiBoZWlnaHQ9IjUwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRp
bnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI25ldFNoYWRvdykiLz4KICAgIDx0ZXh0IHg9
IjE0MCIgeT0iNDUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2Vp
Z2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5BcHBsaWNhdGlvbiBMYXllcjwvdGV4dD4K
ICAgIDx0ZXh0IHg9IjE0MCIgeT0iNjIiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0i
MTQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPkhUVFAvMywgU1NILCBDdXN0b20gQmluYXJ5IFJQQzwv
dGV4dD4KICAgIDwhLS0gVHJhbnNwb3J0IExheWVyIC0tPgogICAgPHJlY3QgeD0iMCIgeT0iODUi
IHdpZHRoPSIyODAiIGhlaWdodD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2
YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tl
LXdpZHRoPSIyIiBmaWx0ZXI9InVybCgjbmV0U2hhZG93KSIvPgogICAgPHRleHQgeD0iMTQwIiB5
PSIxMTAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJi
b2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5UcmFuc3BvcnQgTGF5ZXI8L3RleHQ+CiAgICA8dGV4
dCB4PSIxNDAiIHk9IjEyNyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgdGV4
dC1hbmNob3I9Im1pZGRsZSI+VENQIChTdHJlYW1zKSAvIFVEUCAoRGF0YWdyYW1zKTwvdGV4dD4K
ICAgIDwhLS0gSW50ZXJuZXQgTGF5ZXIgLS0+CiAgICA8cmVjdCB4PSIwIiB5PSIxNTAiIHdpZHRo
PSIyODAiIGhlaWdodD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1z
dmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRo
PSIyIiBmaWx0ZXI9InVybCgjbmV0U2hhZG93KSIvPgogICAgPHRleHQgeD0iMTQwIiB5PSIxNzUi
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0
ZXh0LWFuY2hvcj0ibWlkZGxlIj5JbnRlcm5ldCBMYXllciAoSVApPC90ZXh0PgogICAgPHRleHQg
eD0iMTQwIiB5PSIxOTIiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQt
YW5jaG9yPSJtaWRkbGUiPklQdjQgLyBJUHY2IFJvdXRpbmcgJmFtcDsgQWRkcmVzc2luZzwvdGV4
dD4KICAgIDwhLS0gTGluayBMYXllciAtLT4KICAgIDxyZWN0IHg9IjAiIHk9IjIxNSIgd2lkdGg9
IjI4MCIgaGVpZ2h0PSI1MCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2
Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9
IjIiIGZpbHRlcj0idXJsKCNuZXRTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSIxNDAiIHk9IjI0MCIg
ZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIHRl
eHQtYW5jaG9yPSJtaWRkbGUiPkxpbmsgJmFtcDsgUGh5c2ljYWwgTGF5ZXI8L3RleHQ+CiAgICA8
dGV4dCB4PSIxNDAiIHk9IjI1NyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIg
dGV4dC1hbmNob3I9Im1pZGRsZSI+RXRoZXJuZXQsIFdpLUZpIChGcmFtZXMgJmFtcDsgTUFDcyk8
L3RleHQ+CiAgPC9nPgogIDwhLS0gUmlnaHQ6IEVuY2Fwc3VsYXRpb24gRGF0YSBQaXBlbGluZSAt
LT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSgzNzAsIDQwKSI+CiAgICA8dGV4dCB4PSIyNTAi
IHk9IjAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTYiIGZvbnQtd2VpZ2h0PSJi
b2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5QYWNrZXQgRW5jYXBzdWxhdGlvbiBQcm90b2NvbCBE
YXRhIFVuaXQgKFBEVSk8L3RleHQ+CiAgICA8IS0tIEFwcCBEYXRhIC0tPgogICAgPHJlY3QgeD0i
MjMwIiB5PSIyMCIgd2lkdGg9IjI2MCIgaGVpZ2h0PSI0NSIgcng9IjQiIHN0cm9rZT0idmFyKC0t
c3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1z
dmctdGludC0xLWZhbGxiYWNrKSkiIGZpbHRlcj0idXJsKCNuZXRTaGFkb3cpIi8+CiAgICA8dGV4
dCB4PSIzNjAiIHk9IjQ3IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250
LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+QXBwbGljYXRpb24gTWVzc2FnZSBQ
YXlsb2FkPC90ZXh0PgogICAgPCEtLSBUcmFuc3BvcnQgU2VnbWVudCAtLT4KICAgIDxnIHRyYW5z
Zm9ybT0idHJhbnNsYXRlKDE0MCwgODUpIj4KICAgICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9
IjkwIiBoZWlnaHQ9IjQ1IiByeD0iNCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lk
dGg9IjIiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2sp
KSIgZmlsdGVyPSJ1cmwoI25ldFNoYWRvdykiLz4KICAgICAgPHRleHQgeD0iNDUiIHk9IjI3IiBm
aWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgdGV4
dC1hbmNob3I9Im1pZGRsZSI+VENQIEhkcjwvdGV4dD4KICAgICAgPHJlY3QgeD0iOTAiIHk9IjAi
IHdpZHRoPSIyNjAiIGhlaWdodD0iNDUiIHJ4PSI0IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0
cm9rZS13aWR0aD0iMiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1m
YWxsYmFjaykpIiBmaWx0ZXI9InVybCgjbmV0U2hhZG93KSIvPgogICAgICA8dGV4dCB4PSIyMjAi
IHk9IjI3IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0i
Ym9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+QXBwbGljYXRpb24gTWVzc2FnZSBQYXlsb2FkPC90
ZXh0PgogICAgPC9nPgogICAgPCEtLSBJbnRlcm5ldCBQYWNrZXQgLS0+CiAgICA8ZyB0cmFuc2Zv
cm09InRyYW5zbGF0ZSg3MCwgMTUwKSI+CiAgICAgIDxyZWN0IHg9IjAiIHk9IjAiIHdpZHRoPSI3
MCIgaGVpZ2h0PSI0NSIgcng9IjQiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRo
PSIyIiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSki
IGZpbHRlcj0idXJsKCNuZXRTaGFkb3cpIi8+CiAgICAgIDx0ZXh0IHg9IjM1IiB5PSIyNyIgZmls
bD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQt
YW5jaG9yPSJtaWRkbGUiPklQIEhkcjwvdGV4dD4KICAgICAgPHJlY3QgeD0iNzAiIHk9IjAiIHdp
ZHRoPSI5MCIgaGVpZ2h0PSI0NSIgcng9IjQiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tl
LXdpZHRoPSIyIiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIGZpbHRlcj0idXJsKCNuZXRTaGFkb3cpIi8+CiAgICAgIDx0ZXh0IHg9IjExNSIgeT0i
MjciIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xk
IiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5UQ1AgSGRyPC90ZXh0PgogICAgICA8cmVjdCB4PSIxNjAi
IHk9IjAiIHdpZHRoPSIyNjAiIGhlaWdodD0iNDUiIHJ4PSI0IiBzdHJva2U9InZhcigtLXN2Zy1m
ZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRp
bnQtMS1mYWxsYmFjaykpIiBmaWx0ZXI9InVybCgjbmV0U2hhZG93KSIvPgogICAgICA8dGV4dCB4
PSIyOTAiIHk9IjI3IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdl
aWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+QXBwbGljYXRpb24gTWVzc2FnZSBQYXls
b2FkPC90ZXh0PgogICAgPC9nPgogICAgPCEtLSBFdGhlcm5ldCBGcmFtZSAtLT4KICAgIDxnIHRy
YW5zZm9ybT0idHJhbnNsYXRlKDAsIDIxNSkiPgogICAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0
aD0iNzAiIGhlaWdodD0iNDUiIHJ4PSI0IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13
aWR0aD0iMiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFj
aykpIiBmaWx0ZXI9InVybCgjbmV0U2hhZG93KSIvPgogICAgICA8dGV4dCB4PSIzNSIgeT0iMjci
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0
ZXh0LWFuY2hvcj0ibWlkZGxlIj5FdGggSGRyPC90ZXh0PgogICAgICA8cmVjdCB4PSI3MCIgeT0i
MCIgd2lkdGg9IjcwIiBoZWlnaHQ9IjQ1IiByeD0iNCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBz
dHJva2Utd2lkdGg9IjIiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEt
ZmFsbGJhY2spKSIgZmlsdGVyPSJ1cmwoI25ldFNoYWRvdykiLz4KICAgICAgPHRleHQgeD0iMTA1
IiB5PSIyNyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9
ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPklQIEhkcjwvdGV4dD4KICAgICAgPHJlY3QgeD0i
MTQwIiB5PSIwIiB3aWR0aD0iOTAiIGhlaWdodD0iNDUiIHJ4PSI0IiBzdHJva2U9InZhcigtLXN2
Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3Zn
LXRpbnQtMS1mYWxsYmFjaykpIiBmaWx0ZXI9InVybCgjbmV0U2hhZG93KSIvPgogICAgICA8dGV4
dCB4PSIxODUiIHk9IjI3IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250
LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+VENQIEhkcjwvdGV4dD4KICAgICAg
PHJlY3QgeD0iMjMwIiB5PSIwIiB3aWR0aD0iMjMwIiBoZWlnaHQ9IjQ1IiByeD0iNCIgc3Ryb2tl
PSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbGw9InZhcigtLXN2Zy10aW50LTEs
IHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgZmlsdGVyPSJ1cmwoI25ldFNoYWRvdykiLz4K
ICAgICAgPHRleHQgeD0iMzQ1IiB5PSIyNyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXpl
PSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlBheWxvYWQgRGF0
YTwvdGV4dD4KICAgICAgPHJlY3QgeD0iNDYwIiB5PSIwIiB3aWR0aD0iNTAiIGhlaWdodD0iNDUi
IHJ4PSI0IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsbD0idmFy
KC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBmaWx0ZXI9InVybCgj
bmV0U2hhZG93KSIvPgogICAgICA8dGV4dCB4PSI0ODUiIHk9IjI3IiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRs
ZSI+Q1JDPC90ZXh0PgogICAgPC9nPgogIDwvZz4KPC9zdmc+
" alt="Figure 17–2: TCP/IP Network Stack" />
<figcaption>Figure 17–2: TCP/IP Network Stack</figcaption>
</figure>

**1. The Link Layer (Physical / Data Link):**

Operates directly over physical hardware (Ethernet cards, 802.11 Wi-Fi
chips). Communication is local, addressing nodes within the same
physical broadcast domain using 48-bit **MAC (Media Access Control)
addresses**. Units of transmission are called **Frames**.

**2. The Internet Layer (IP):** Provides host-to-host packet routing
across heterogeneous, globally distributed networks using logical **IP
Addresses** (32-bit IPv4 or 128-bit IPv6). The Internet Protocol is
strictly:

- **Connectionless:** Packets are routed independently without
  pre-allocating a dedicated physical circuit.
- **Unreliable (Best Effort):** IP provides no delivery guarantees.
  Packets may be dropped by congested intermediate routers, duplicated,
  or arrive completely out of order.

**3. The Transport Layer (TCP vs UDP):**

While IP routes packets between physical machines, the Transport Layer
multiplexes communication between discrete *processes* on those machines
using 16-bit **Port numbers** ($0 \text{ to } 65535$).

- **UDP (User Datagram Protocol):** A minimal wrapper around IP. It adds
  process port numbers and an optional payload checksum. UDP is
  lightweight, message-oriented, and preserves datagram boundaries, but
  does not guarantee packet delivery or arrival order. It is ideal for
  real-time video streaming, DNS lookups, and multiplayer games.
- **TCP (Transmission Control Protocol):** Transforms an unreliable,
  packet-dropping IP network into a reliable, bidirectional, ordered,
  flow-controlled byte stream.

The features of the UDP and TCP protocols are compared in Table 17–1.

| Feature | TCP (Transmission Control Protocol) | UDP (User Datagram Protocol) |
|------------------------|------------------------|------------------------|
| **Connection Model** | Connection-oriented (3-way handshake) | Connectionless |
| **Delivery Guarantee** | Guaranteed via ACKs and retransmissions | Best-effort (packets may be dropped) |
| **Ordering** | Strict sequential ordering via sequence numbers | Out-of-order delivery possible |
| **Framing Concept** | Unbounded, continuous byte stream | Discrete datagram packets |
| **Flow and Congestion Control** | Dynamic sliding window and backoff algorithms | None (Application must manage congestion) |
| **Header Overhead** | 20 to 60 bytes | Exactly 8 bytes |

Table 17–1: Comparison between TCP and UDP {#tbl:tcp-udp}

### 17.3 Reliable Delivery over Unreliable Channels

How does TCP manufacture absolute reliability out of an untrusted
medium?

#### 17.3.1 The Two Generals' Problem

Database and network theory proves that achieving guaranteed mutual
consensus over an unreliable communications link is mathematically
impossible.

Imagine two generals, $G_1$ and $G_2$, positioned on opposing hills
preparing to attack a fortified enemy valley. If both attack
simultaneously, they conquer the valley; if either attacks alone, their
army is annihilated. $G_1$ sends a messenger across the valley to $G_2$:
*"Attack at dawn"*.

- Did the messenger get captured? $G_1$ cannot know without receiving an
  acknowledgement (ACK) from $G_2$.
- If $G_2$ sends an ACK: did the ACK reach $G_1$? $G_2$ cannot attack
  until they know $G_1$ received the ACK, requiring an ACK-to-the-ACK.
- This recursive acknowledgement dilemma continues infinitely.

TCP solves this practical limitation not with absolute mathematical
certainty, but through pragmatic state machines, finite retransmission
timeouts (RTO), and explicit connection handshakes (see Figure 17–3).

<figure id="fig:client-server">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAzMCA4MDAgMjgwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyODAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iZ2xvd1NoYWRvdyIgeD0iLTEwJSIgeT0i
LTEwJSIgd2lkdGg9IjEyMCUiIGhlaWdodD0iMTMwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9
IjAiIGR5PSIyIiBzdGREZXZpYXRpb249IjMiIGZsb29kLWNvbG9yPSIjMDAwIiBmbG9vZC1vcGFj
aXR5PSIwLjQiLz4KICAgIDwvZmlsdGVyPgogIDwvZGVmcz4KICA8IS0tIENsaWVudCAmIFNlcnZl
ciBUaW1lbGluZSBBeGVzIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDgwLCAzMCkiPgog
ICAgPCEtLSBDbGllbnQgVGltZWxpbmUgLS0+CiAgICA8bGluZSB4MT0iMTIwIiB5MT0iNDAiIHgy
PSIxMjAiIHkyPSIyNzAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIzIiBz
dHJva2UtZGFzaGFycmF5PSI0LDQiLz4KICAgIDxyZWN0IHg9IjUwIiB5PSIxMCIgd2lkdGg9IjE0
MCIgaGVpZ2h0PSIzMCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10
aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIi
Lz4KICAgIDx0ZXh0IHg9IjEyMCIgeT0iMzAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6
ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5DbGllbnQgKEhv
c3QgQSk8L3RleHQ+CiAgICA8IS0tIFNlcnZlciBUaW1lbGluZSAtLT4KICAgIDxsaW5lIHgxPSI2
MDAiIHkxPSI0MCIgeDI9IjYwMCIgeTI9IjI3MCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJv
a2Utd2lkdGg9IjMiIHN0cm9rZS1kYXNoYXJyYXk9IjQsNCIvPgogICAgPHJlY3QgeD0iNTMwIiB5
PSIxMCIgd2lkdGg9IjE0MCIgaGVpZ2h0PSIzMCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50
LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBz
dHJva2Utd2lkdGg9IjIiLz4KICAgIDx0ZXh0IHg9IjYwMCIgeT0iMzAiIGZpbGw9InZhcigtLXN2
Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIj5TZXJ2ZXIgKEhvc3QgQik8L3RleHQ+CiAgICA8IS0tIEhhbmRzaGFrZSBTdGVwIDE6IFNZ
TiAtLT4KICAgIDxwYXRoIGQ9Ik0gMTIwIDcwIEwgNjAwIDExMCIgc3Ryb2tlPSJ2YXIoLS1zdmct
ZmcpIiBzdHJva2Utd2lkdGg9IjIuNSIgZmlsbD0ibm9uZSIvPgogICAgPHBvbHlnb24gcG9pbnRz
PSI2MDAsMTEwIDU5MiwxMDMgNTkxLDExNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRl
eHQgeD0iMzYwIiB5PSI3NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgZm9u
dC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPjEuIFNZTiBbU2VxID0gMTAwMF08
L3RleHQ+CiAgICA8IS0tIEhhbmRzaGFrZSBTdGVwIDI6IFNZTi1BQ0sgLS0+CiAgICA8cGF0aCBk
PSJNIDYwMCAxMzAgTCAxMjAgMTcwIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0
aD0iMi41IiBmaWxsPSJub25lIi8+CiAgICA8cG9seWdvbiBwb2ludHM9IjEyMCwxNzAgMTI4LDE2
MyAxMjksMTc0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiAvPgogICAgPHRleHQgeD0iMzYwIiB5PSIx
MzAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xk
IiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj4yLiBTWU4tQUNLIFtTZXEgPSA1MDAwLCBBY2sgPSAxMDAx
XTwvdGV4dD4KICAgIDwhLS0gSGFuZHNoYWtlIFN0ZXAgMzogQUNLIC0tPgogICAgPHBhdGggZD0i
TSAxMjAgMTkwIEwgNjAwIDIzMCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9
IjIuNSIgZmlsbD0ibm9uZSIvPgogICAgPHBvbHlnb24gcG9pbnRzPSI2MDAsMjMwIDU5MiwyMjMg
NTkxLDIzNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMzYwIiB5PSIxOTAi
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0
ZXh0LWFuY2hvcj0ibWlkZGxlIj4zLiBBQ0sgW1NlcSA9IDEwMDEsIEFjayA9IDUwMDFdPC90ZXh0
PgogICAgPCEtLSBTdGF0ZSBMYWJlbHMgLS0+CiAgICA8dGV4dCB4PSI1MCIgeT0iNzUiIGZpbGw9
InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlNZTl9T
RU5UPC90ZXh0PgogICAgPHRleHQgeD0iNjcwIiB5PSIxMjUiIGZpbGw9InZhcigtLXN2Zy1mZyki
IGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlNZTl9SQ1ZEPC90ZXh0PgogICAg
PHRleHQgeD0iNTAiIHk9IjE5NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIg
Zm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPkVTVEFCTElTSEVEPC90ZXh0
PgogICAgPHRleHQgeD0iNjcwIiB5PSIyMzUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6
ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5FU1RBQkxJU0hF
RDwvdGV4dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Figure 17–3: Three-Way TCP Handshake" />
<figcaption>Figure 17–3: Three-Way TCP Handshake</figcaption>
</figure>

#### 17.3.2 Sequence Numbers and Cumulative Acknowledgements

Every byte transmitted over a TCP connection is assigned a 32-bit
sequence number. If a client transmits a segment of 500 bytes starting
at `Seq = 1001`, the server acknowledges receipt with `Ack = 1501`
(indicating: *"I have received everything up to byte 1500; I am
expecting byte 1501 next"*).

If the transmission timer expires before the sender receives the ACK,
the unacknowledged segment is retransmitted. If packets arrive scrambled
or out of order due to dynamic IP routing, the receiver uses the
sequence numbers to slot the segments into the correct contiguous order
before presenting them to the application layer.

### 17.4 The Berkeley Sockets API: The Unix Abstraction

In Unix-derived operating systems, network communication is surfaced
through the **Berkeley Sockets API**. Sockets adhere to the classic Unix
design principle: **"Everything is a file descriptor"**.

A socket descriptor is an integer referencing an operating system kernel
data structure. Reading from a network connection uses the same
low-level descriptor semantics as reading bytes from a local hard drive
or console stream. See Figure 17–4 for a diagrammatical representation
of client-server communication using sockets.

<figure id="fig:sockets">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MjIuNjAwMDQi
IGhlaWdodD0iNDI1LjIiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNTIyLjYwMDA0IDQy
NS4yIj4KICA8c3R5bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xl
LCAuc3ZnYm9iIHJlY3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJv
a2Utd2lkdGg6IDEuNTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAg
c3Ryb2tlLWxpbmVjYXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdi
b2IgdGV4dCB7CiAgd2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWls
eTogQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6
IDEzcHg7Cn0KCi5zdmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6
IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7
Cn0KCi5zdmdib2IgLmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxl
ZCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5u
b2ZpbGwgewogIGZpbGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93
IHsKICBtYXJrZXItZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fy
cm93IHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9kaWFtb25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUp
Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwo
I29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBt
YXJrZXItc3RhcnQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Jp
Z19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVy
bCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9
ImFycm93IiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRo
PSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAg
PHBvbHlnb24gcG9pbnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2Vy
PgogICAgPG1hcmtlciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiBy
ZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+
PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjIiIGNsYXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJvcGVuX2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+
PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtl
cj4KICA8L2RlZnM+CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9
IjUyMi42MDAwNCIgaGVpZ2h0PSI0MjUuMiI+PC9yZWN0PgogIDxyZWN0IHg9IjUwLjciIHk9IjM2
Ni42IiB3aWR0aD0iMTE3IiBoZWlnaHQ9IjMxLjIwMDAxMiIgY2xhc3M9InNvbGlkIG5vZmlsbCIg
cng9IjAiPjwvcmVjdD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3OS45NTAwMDUi
IHk9IjM4Ni4xIiA+Y2xvc2U8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MTE4Ljk1MDAwNSIgeT0iMzg2LjEiID4oKTwvdGV4dD4KICA8cmVjdCB4PSIzOTMuOTAwMDIiIHk9
IjM2Ni42IiB3aWR0aD0iMTE3IiBoZWlnaHQ9IjMxLjIwMDAxMiIgY2xhc3M9InNvbGlkIG5vZmls
bCIgcng9IjAiPjwvcmVjdD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MjMuMTUw
MDIiIHk9IjM4Ni4xIiA+Y2xvc2U8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iNDYyLjE1MDAyIiB5PSIzODYuMSIgPigpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjU2LjU1MDAwMyIgeT0iMTEuNzAwMDAxIiA+U2VydmVyPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExMS4xNSIgeT0iMTEuNzAwMDAxIiA+UHJvY2Vzczwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzOTkuNzUiIHk9IjExLjcwMDAw
MSIgPkNsaWVudDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NTQuMzUi
IHk9IjExLjcwMDAwMSIgPlByb2Nlc3M8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iNzkuOTUwMDA1IiB5PSI0Mi45IiA+c29ja2V0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjQyMy4xNTAwMiIgeT0iNDIuOSIgPnNvY2tldDwvdGV4dD4KICA8cG9s
eWdvbiBwb2ludHM9IjEwOS4yMDAwMDUsODEuOSAxMTcsODEuOSAxMTMuMTAwMDA2LDkzLjYwMDAw
NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9Ijg3Ljc1IiB5PSIxMjAuOSIgPmJpbmQ8L3RleHQ+CiAgPHBvbHlnb24gcG9pbnRzPSIxMDku
MjAwMDA1LDE1OS45MDAwMSAxMTcsMTU5LjkwMDAxIDExMy4xMDAwMDYsMTcxLjYiIGNsYXNzPSJm
aWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3OS45NTAw
MDUiIHk9IjE5OC45MDAwMSIgPmxpc3RlbjwvdGV4dD4KICA8cG9seWdvbiBwb2ludHM9IjEwOS4y
MDAwMDUsMjM3LjkwMDAxIDExNywyMzcuOTAwMDEgMTEzLjEwMDAwNiwyNDkuNiIgY2xhc3M9ImZp
bGxlZCI+PC9wb2x5Z29uPgogIDxwb2x5Z29uIHBvaW50cz0iNDUyLjQwMDAyLDIzNy45MDAwMSA0
NjAuMiwyMzcuOTAwMDEgNDU2LjMwMDAyLDI0OS42IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzkuOTUwMDA1IiB5PSIyNzYuOSIgPmFj
Y2VwdDwvdGV4dD4KICA8cG9seWdvbiBwb2ludHM9IjE3OS40MDAwMSwyNjkuMSAxNzEuNiwyNzMg
MTc5LjQwMDAxLDI3Ni45IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjE3
OS40MDAwMSIgeTE9IjI3MyIgeDI9IjM4Mi4yIiB5Mj0iMjczIiBjbGFzcz0ic29saWQiPjwvbGlu
ZT4KICA8cG9seWdvbiBwb2ludHM9IjM4Mi4yLDI2OS4xIDM5MCwyNzMgMzgyLjIsMjc2LjkiIGNs
YXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0
MTUuMzUiIHk9IjI3Ni45IiA+Y29ubmVjdDwvdGV4dD4KICA8cG9seWdvbiBwb2ludHM9IjEwOS4y
MDAwMDUsMzQ3LjEgMTE3LDM0Ny4xIDExMy4xMDAwMDYsMzU4LjgwMDAyIiBjbGFzcz0iZmlsbGVk
Ij48L3BvbHlnb24+CiAgPHBvbHlnb24gcG9pbnRzPSI0NTIuNDAwMDIsMzQ3LjEgNDYwLjIsMzQ3
LjEgNDU2LjMwMDAyLDM1OC44MDAwMiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxwb2x5
Z29uIHBvaW50cz0iMTMyLjYsMzE1LjkgMTI0LjgsMzE5LjgwMDAyIDEzMi42LDMyMy43IiBjbGFz
cz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjEzMi42IiB5MT0iMzE5LjgwMDAyIiB4
Mj0iMTc5LjQwMDAxIiB5Mj0iMzE5LjgwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxODkuMTUwMDEiIHk9IjMyMy43IiA+QXBwbGljYXRp
b248L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjgyLjc1IiB5PSIzMjMu
NyIgPkRhdGE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzIxLjc1IiB5
PSIzMjMuNyIgPkV4Y2hhbmdlPC90ZXh0PgogIDxsaW5lIHgxPSIzOTAiIHkxPSIzMTkuODAwMDIi
IHgyPSI0MzYuODAwMDIiIHkyPSIzMTkuODAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxw
b2x5Z29uIHBvaW50cz0iNDM2LjgwMDAyLDMxNS45IDQ0NC42LDMxOS44MDAwMiA0MzYuODAwMDIs
MzIzLjciIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxMjYuNzUiIHk9IjQyLjkiID4oKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSI0NjkuOTUiIHk9IjQyLjkiID4oKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIxMTguOTUwMDA1IiB5PSIxMjAuOSIgPigpPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjEyNi43NSIgeT0iMTk4LjkwMDAxIiA+KCk8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjIwLjM1IiB5PSIyNjEuMzAwMDIiID4zLVdh
eSBIYW5kc2hha2U8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTI2Ljc1
IiB5PSIyNzYuOSIgPigpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ2
OS45NSIgeT0iMjc2LjkiID4oKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIxOTYuOTUwMDEiIHk9IjMzOS4zMDAwMiIgPnNlbmQoKSAvIHJlY3YoKSBzdHJlYW08L3RleHQ+
CiAgPGc+CiAgICA8bGluZSB4MT0iNTAuNyIgeTE9IjIzLjQwMDAwMiIgeDI9IjE2Ny43IiB5Mj0i
MjMuNDAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSI1MC43IiB5MT0i
MjMuNDAwMDAyIiB4Mj0iNTAuNyIgeTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+
CiAgICA8bGluZSB4MT0iMTY3LjciIHkxPSIyMy40MDAwMDIiIHgyPSIxNjcuNyIgeTI9IjU0LjYw
MDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iNTAuNyIgeTE9IjU0LjYw
MDAwMiIgeDI9IjE2Ny43IiB5Mj0iNTQuNjAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAg
IDxsaW5lIHgxPSIxMTMuMTAwMDA2IiB5MT0iNTQuNjAwMDAyIiB4Mj0iMTEzLjEwMDAwNiIgeTI9
IjgxLjkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4KICA8Zz4KICAgIDxsaW5lIHgxPSIz
OTMuOTAwMDIiIHkxPSIyMy40MDAwMDIiIHgyPSI1MTAuOTAwMDIiIHkyPSIyMy40MDAwMDIiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjM5My45MDAwMiIgeTE9IjIzLjQwMDAw
MiIgeDI9IjM5My45MDAwMiIgeTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAg
ICA8bGluZSB4MT0iNTEwLjkwMDAyIiB5MT0iMjMuNDAwMDAyIiB4Mj0iNTEwLjkwMDAyIiB5Mj0i
NTQuNjAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIzOTMuOTAwMDIi
IHkxPSI1NC42MDAwMDIiIHgyPSI1MTAuOTAwMDIiIHkyPSI1NC42MDAwMDIiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjQ1Ni4zMDAwMiIgeTE9IjU0LjYwMDAwMiIgeDI9IjQ1
Ni4zMDAwMiIgeTI9IjIzNy45MDAwMSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPC9nPgogIDxn
PgogICAgPGxpbmUgeDE9IjUwLjciIHkxPSIxMDEuNCIgeDI9IjE2Ny43IiB5Mj0iMTAxLjQiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjUwLjciIHkxPSIxMDEuNCIgeDI9IjUw
LjciIHkyPSIxMzIuNiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTY3Ljci
IHkxPSIxMDEuNCIgeDI9IjE2Ny43IiB5Mj0iMTMyLjYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
ICAgPGxpbmUgeDE9IjUwLjciIHkxPSIxMzIuNiIgeDI9IjE2Ny43IiB5Mj0iMTMyLjYiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjExMy4xMDAwMDYiIHkxPSIxMzIuNiIgeDI9
IjExMy4xMDAwMDYiIHkyPSIxNTkuOTAwMDEiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4K
ICA8Zz4KICAgIDxsaW5lIHgxPSI1MC43IiB5MT0iMTc5LjQwMDAxIiB4Mj0iMTY3LjciIHkyPSIx
NzkuNDAwMDEiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjUwLjciIHkxPSIx
NzkuNDAwMDEiIHgyPSI1MC43IiB5Mj0iMjEwLjYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAg
PGxpbmUgeDE9IjE2Ny43IiB5MT0iMTc5LjQwMDAxIiB4Mj0iMTY3LjciIHkyPSIyMTAuNiIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iNTAuNyIgeTE9IjIxMC42IiB4Mj0iMTY3
LjciIHkyPSIyMTAuNiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTEzLjEw
MDAwNiIgeTE9IjIxMC42IiB4Mj0iMTEzLjEwMDAwNiIgeTI9IjIzNy45MDAwMSIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPC9nPgogIDxnPgogICAgPGxpbmUgeDE9IjUwLjciIHkxPSIyNTcuNCIg
eDI9IjE2Ny43IiB5Mj0iMjU3LjQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9
IjUwLjciIHkxPSIyNTcuNCIgeDI9IjUwLjciIHkyPSIyODguNiIgY2xhc3M9InNvbGlkIj48L2xp
bmU+CiAgICA8bGluZSB4MT0iMTY3LjciIHkxPSIyNTcuNCIgeDI9IjE2Ny43IiB5Mj0iMjg4LjYi
IGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjUwLjciIHkxPSIyODguNiIgeDI9
IjE2Ny43IiB5Mj0iMjg4LjYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjEx
My4xMDAwMDYiIHkxPSIyODguNiIgeDI9IjExMy4xMDAwMDYiIHkyPSIzNDcuMSIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPC9nPgogIDxnPgogICAgPGxpbmUgeDE9IjM5My45MDAwMiIgeTE9IjI1
Ny40IiB4Mj0iNTEwLjkwMDAyIiB5Mj0iMjU3LjQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAg
PGxpbmUgeDE9IjM5My45MDAwMiIgeTE9IjI1Ny40IiB4Mj0iMzkzLjkwMDAyIiB5Mj0iMjg4LjYi
IGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjUxMC45MDAwMiIgeTE9IjI1Ny40
IiB4Mj0iNTEwLjkwMDAyIiB5Mj0iMjg4LjYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxp
bmUgeDE9IjM5My45MDAwMiIgeTE9IjI4OC42IiB4Mj0iNTEwLjkwMDAyIiB5Mj0iMjg4LjYiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjQ1Ni4zMDAwMiIgeTE9IjI4OC42IiB4
Mj0iNDU2LjMwMDAyIiB5Mj0iMzQ3LjEiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4KPC9z
dmc+Cg==
" alt="" />
<figcaption>Figure 17–4: Server-Client Socket-Based
Communication</figcaption>
</figure>

### The Sockets Lifecycle

1.  `socket(domain, type, protocol)`: Requests a new communication
    endpoint from the operating system kernel.
    - `AF_INET` (IPv4) or `AF_INET6` (IPv6).
    - `SOCK_STREAM` (TCP) or `SOCK_DGRAM` (UDP).
2.  `bind(sock_fd, address, addr_len)`: Binds the socket to a specific
    local network interface and port number.
3.  `listen(sock_fd, backlog)`: Transitions a TCP socket into a passive
    state, ready to receive incoming client connection requests. The
    `backlog` parameter sets the queue length for pending, unaccepted
    connections.
4.  `accept(sock_fd, client_addr, addr_len)`: **Blocks** the calling
    thread until a remote client connects. Upon connection, it returns a
    **brand-new socket descriptor** dedicated exclusively to that
    specific client session, leaving the original listening socket free
    to accept further incoming connections.
5.  `connect(sock_fd, server_addr, addr_len)`: Initiates the TCP 3-way
    handshake with a remote listening server.
6.  `send()` / `recv()`: Writes bytes into and reads bytes from the
    kernel's network transmission buffers.

### 17.5 The Engineering Problem: The "Sticky Packet" Fallacy

One of the most frequent bugs written by engineers building distributed
software is treating TCP as a **message protocol** rather than a **byte
stream protocol**.

If your client runs:

``` cpp
send(sock, "HELLO", 5, 0);
send(sock, "WORLD", 5, 0);
```

The receiving application is **not** guaranteed to read `"HELLO"` in the
first `recv()` call and `"WORLD"` in the second (see Figure 17–5 for two
different possible scenarios).

TCP knows nothing about your application-level boundaries (see
Figure 17–5). Depending on Maximum Segment Size (MSS), Ethernet MTU,
Nagle's algorithm, and operating system scheduling, the receiver's
`recv()` call might return:

- A single 10-byte buffer: `"HELLOWORLD"` (Packet Coalescing / Sticky
  Packet).
- Three fragmented chunks: `"HEL"`, then `"LOWO"`, then `"RLD"` (Packet
  Fragmentation).
- Ten individual 1-byte read events.

<figure id="fig:packets">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MTYuMiIgaGVp
Z2h0PSIxNzUuNiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2MTYuMiAxNzUuNiI+CiAg
PHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2Jv
YiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRo
OiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1s
aW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQg
ewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9
Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3Bh
cmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3Zn
Ym9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZp
bGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFy
a2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25k
IHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRf
bWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5z
dGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2Np
cmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0
YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9j
aXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19v
cGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIg
dmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFy
a2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29u
IHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxt
YXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdv
bj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4
IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmll
bnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIy
IiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0i
b3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldp
ZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAg
ICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xl
PgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9k
ZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2MTYuMiIg
aGVpZ2h0PSIxNzUuNiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijku
NzUiIHk9IjExLjcwMDAwMSIgPlNlbmRlcjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI2NC4zNSIgeT0iMTEuNzAwMDAxIiA+QnVmZmVyOjwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIxNzMuNTUiIHk9IjExLjcwMDAwMSIgPls8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTg5LjE1MDAxIiB5PSIxMS43MDAwMDEiID5IPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIwNC43NSIgeT0iMTEuNzAwMDAx
IiA+RTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMjAuMzUiIHk9IjEx
LjcwMDAwMSIgPkw8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjM1Ljk1
MDAxIiB5PSIxMS43MDAwMDEiID5MPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjI1MS41NSIgeT0iMTEuNzAwMDAxIiA+TzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIyNjcuMTUiIHk9IjExLjcwMDAwMSIgPl08L3RleHQ+CiAgPGxpbmUgeDE9IjI2
MS4zMDAwMiIgeTE9IjE1LjYiIHgyPSIyNjEuMzAwMDIiIHkyPSIzNS4xMDAwMDIiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMjU3LjQsMzUuMTAwMDAyIDI2NS4yLDM1
LjEwMDAwMiAyNjEuMzAwMDIsNDYuODAwMDAzIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAg
PGxpbmUgeDE9IjE2My44IiB5MT0iNTQuNjAwMDAyIiB4Mj0iNDM2LjgwMDAyIiB5Mj0iNTQuNjAw
MDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjQzNi44MDAwMiw1
MC43IDQ0NC42LDU0LjYwMDAwMiA0MzYuODAwMDIsNTguNSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5
Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE2NS43NSIgeT0iNzQuMSIgPi4u
LjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxOTYuOTUwMDEiIHk9Ijc0
LjEiID4weDQ4PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIzNS45NTAw
MSIgeT0iNzQuMSIgPjB4NDU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
Mjc0Ljk1IiB5PSI3NC4xIiA+MHg0QzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIzMTMuOTUiIHk9Ijc0LjEiID4weDRDPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjM1Mi45NSIgeT0iNzQuMSIgPjB4NEY8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMzkxLjk1IiB5PSI3NC4xIiA+MHg1NzwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSI0MzAuOTUiIHk9Ijc0LjEiID4weDRGPC90ZXh0PgogIDxsaW5l
IHgxPSIyNjEuMzAwMDIiIHkxPSI3OCIgeDI9IjI2MS4zMDAwMiIgeTI9Ijk3LjUiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMjU3LjQsOTcuNSAyNjUuMiw5Ny41IDI2
MS4zMDAwMiwxMDkuMjAwMDA1IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjgyLjc1IiB5PSIxMS43MDAwMDEiID5bPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5OC4zNSIgeT0iMTEuNzAwMDAxIiA+VzwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMTMuOTUiIHk9IjExLjcwMDAwMSIg
Pk88L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzI5LjU1MDAyIiB5PSIx
MS43MDAwMDEiID5SPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM0NS4x
NSIgeT0iMTEuNzAwMDAxIiA+TDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIzNjAuNzUiIHk9IjExLjcwMDAwMSIgPkQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzc2LjM1IiB5PSIxMS43MDAwMDEiID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9IjU4LjUiID5OZXR3b3JrPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjcyLjE1IiB5PSI1OC41IiA+U3RyZWFtOjwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NjkuOTUiIHk9Ijc0LjEiID4weDUyPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUwOC45NSIgeT0iNzQuMSIgPjB4NEM8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTQ3Ljk1IiB5PSI3NC4xIiA+
MHg0NDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1ODYuOTUiIHk9Ijc0
LjEiID4uLi48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43NSIgeT0i
MTIwLjkiID5SZWNlaXZlcjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3
OS45NTAwMDUiIHk9IjEyMC45IiA+U2NlbmFyaW9zOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIxNy41NTAwMDEiIHk9IjEzNi41IiA+U2NlbmFyaW88L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iODcuNzUiIHk9IjEzNi41IiA+QTwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMTguOTUwMDA1IiB5PSIxMzYuNSIgPls8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTM0LjU1IiB5PSIxMzYuNSIgPkg8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTUwLjE1MDAxIiB5PSIxMzYu
NSIgPkU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTcuNTUwMDAxIiB5
PSIxNTIuMSIgPlNjZW5hcmlvPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
Ijg3Ljc1IiB5PSIxNTIuMSIgPkI8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMTE4Ljk1MDAwNSIgeT0iMTUyLjEiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEzNC41NSIgeT0iMTUyLjEiID5IPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE1MC4xNTAwMSIgeT0iMTUyLjEiID5FPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjE2NS43NSIgeT0iMTM2LjUiID5MPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjE2NS43NSIgeT0iMTUyLjEiID5MPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE4MS4zNSIgeT0iMTM2LjUiID5MPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE4MS4zNSIgeT0iMTUyLjEiID5dPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE5Ni45NTAwMSIgeT0iMTM2LjUiID5PPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIxMi41NSIgeT0iMTM2LjUiID5X
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIyOC4xNTAwMSIgeT0iMTM2
LjUiID5PPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0My43NSIgeT0i
MTM2LjUiID5SPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1OS4zNSIg
eT0iMTM2LjUiID5MPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI3NC45
NSIgeT0iMTM2LjUiID5EPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5
MC41NTAwMiIgeT0iMTM2LjUiID5dPC90ZXh0PgogIDxsaW5lIHgxPSIxOTUiIHkxPSIxNDguMiIg
eDI9IjIwMi44IiB5Mj0iMTQ4LjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBv
aW50cz0iMjAyLjgsMTQ0LjMgMjEwLjYsMTQ4LjIgMjAyLjgsMTUyLjEiIGNsYXNzPSJmaWxsZWQi
PjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMjAuMzUiIHk9IjE1
Mi4xIiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMzUuOTUwMDEi
IHk9IjE1Mi4xIiA+TDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNTEu
NTUiIHk9IjE1Mi4xIiA+TzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIy
NjcuMTUiIHk9IjE1Mi4xIiA+VzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIyODIuNzUiIHk9IjE1Mi4xIiA+TzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIyOTguMzUiIHk9IjE1Mi4xIiA+XTwvdGV4dD4KICA8bGluZSB4MT0iMzEyIiB5MT0iMTQ4
LjIiIHgyPSIzMTkuODAwMDIiIHkyPSIxNDguMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBv
bHlnb24gcG9pbnRzPSIzMTkuODAwMDIsMTQ0LjMgMzI3LjYsMTQ4LjIgMzE5LjgwMDAyLDE1Mi4x
IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMzM3LjM1IiB5PSIxNTIuMSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iMzUyLjk1IiB5PSIxNTIuMSIgPlI8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzY4LjU1MDAyIiB5PSIxNTIuMSIgPkw8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMzg0LjE1MDAyIiB5PSIxNTIuMSIgPkQ8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzk5Ljc1IiB5PSIxNTIuMSIgPl08L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTUuNTUiIHk9IjEzNi41IiA+OjwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MTUuMzUiIHk9IjEzNi41IiA+KENvYWxlc2NlZCBp
bnRvIDEgcmVhZCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTUuNTUi
IHk9IjE1Mi4xIiA+OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MTUu
MzUiIHk9IjE1Mi4xIiA+KFNwbGl0IGFjcm9zcyAzIHJlYWRzKTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 17–5: Packet Transmission</figcaption>
</figure>

### Application-Layer Framing Strategies

To reliably reconstruct messages across a stream, the application
protocol must define an explicit framing scheme:

1.  **Delimiter-Based Framing:** Messages are terminated by an
    agreed-upon sentinel character or sequence (e.g., `\n`, or HTTP's
    `\r\n\r\n`). *Limitation:* The delimiter cannot appear inside the
    payload unless escaped.
2.  **Fixed-Length Framing:** Every message is precisely $K$ bytes.
    *Limitation:* Wastes bandwidth with padding for small messages;
    cannot support variable payloads.
3.  **Length-Prefixed Framing (Header + Payload / TLV):** Every message
    begins with a fixed-size binary header specifying the exact length
    of the payload that follows. This is the industry standard for
    high-performance, robust binary protocols.

### 17.6 Modern C++ Sockets and Protocol Architecture

The aim is to construct an idiomatic, high-performance, multi-threaded
networking library in Modern C++20.

We will design a complete binary protocol (Figure 17–6) featuring:

- **RAII Socket Encapsulation:** Zero-leak kernel descriptor lifecycle
  management.
- **Length-Prefixed Binary Framing:** Explicit message types, magic byte
  validation, and payload length headers.
- **Endian-Safe Serialisation:** Network byte order conversion for
  cross-platform data interchange.
- **Multi-Threaded Worker Execution:** Leveraging C++20 `std::jthread`
  and `std::stop_token` to manage concurrent client connections safely.

<figure id="fig:protocol">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MTYuMiIgaGVp
Z2h0PSIxMTMuMjAwMDA1IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDYxNi4yIDExMy4y
MDAwMDUiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6
ZTogMTNweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmls
bDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTog
ODsKfQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmls
bGVkIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2Ig
Lm5vZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRf
YXJyb3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2
Z2JvYiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNs
ZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVy
bCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewog
IG1hcmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
YmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0K
Ci5zdmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDog
dXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBp
ZD0iYXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lk
dGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAg
ICA8cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJr
ZXI+CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQi
IHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1z
dGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwy
Ij48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0Jv
eD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdo
dD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBj
eT0iNCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxt
YXJrZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0
IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2
ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVk
Ij48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xl
IiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFy
a2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUg
Y3g9IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFy
a2VyPgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0
aD0iNjE2LjIiIGhlaWdodD0iMTEzLjIwMDAwNSI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE4OS4xNTAwMSIgeT0iMjcuMzAwMDAxIiA+UHJvdG9jb2w8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjU5LjM1IiB5PSIyNy4zMDAwMDEiID5GcmFt
ZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMDYuMTUiIHk9IjI3LjMw
MDAwMSIgPkJpbmFyeTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNjAu
NzUiIHk9IjI3LjMwMDAwMSIgPkxheW91dDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIyNS4zNSIgeT0iNTguNSIgPk1hZ2ljPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE4MS4zNSIgeT0iNTguNSIgPk1zZzwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIyMTIuNTUiIHk9IjU4LjUiID5UeXBlPC90ZXh0PgogIDxsaW5lIHgx
PSIzNDcuMSIgeTE9IjQ2LjgwMDAwMyIgeDI9IjM0Ny4xIiB5Mj0iNzgiIGNsYXNzPSJzb2xpZCI+
PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2MC43NSIgeT0iNTguNSIg
Pkxlbmd0aDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MjQuNTUiIHk9
IjU4LjUiID5QYXlsb2FkPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU2
LjU1MDAwMyIgeT0iNzQuMSIgPjB4Q0FGRTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxODkuMTUwMDEiIHk9Ijc0LjEiID4weDAwMDE8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMzg0LjE1MDAyIiB5PSI3NC4xIiA+MHgwMDAwMDAxQzwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3Mi4xNSIgeT0iNTguNSIgPigyIEJ5dGVz
KTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNTEuNTUiIHk9IjU4LjUi
ID4oMiBCeXRlcyk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDE1LjM1
IiB5PSI1OC41IiA+KDQgQnl0ZXMpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjI0My43NSIgeT0iNzQuMSIgPihFWEVDX1NRTCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNTE2Ljc1IiB5PSI3NC4xIiA+KE4gQnl0ZXMpPC90ZXh0PgogIDxnPgog
ICAgPGxpbmUgeDE9IjExLjcwMDAwMSIgeTE9IjcuOCIgeDI9IjYwNC41IiB5Mj0iNy44IiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIxMS43MDAwMDEiIHkxPSI3LjgiIHgyPSIx
MS43MDAwMDEiIHkyPSI4NS44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSI2
MDQuNSIgeTE9IjcuOCIgeDI9IjYwNC41IiB5Mj0iODUuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+
CiAgICA8bGluZSB4MT0iMTEuNzAwMDAxIiB5MT0iMzkiIHgyPSI2MDQuNSIgeTI9IjM5IiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIxNjcuNyIgeTE9IjM5IiB4Mj0iMTY3Ljci
IHkyPSI4NS44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSI1MDMuMSIgeTE9
IjM5IiB4Mj0iNTAzLjEiIHkyPSI4NS44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5l
IHgxPSIxMS43MDAwMDEiIHkxPSI4NS44IiB4Mj0iNjA0LjUiIHkyPSI4NS44IiBjbGFzcz0ic29s
aWQiPjwvbGluZT4KICA8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 17–6: Protocol Architecture</figcaption>
</figure>

- Listing 17–1 wraps raw POSIX socket descriptors in a clean, move-only
  RAII abstraction.
- Listing 17–2 implements the structured framing protocol. To guarantee
  that an x86 little-endian machine can talk reliably to an ARM or
  big-endian architecture, all multi-byte header integers are converted
  into **Network Byte Order (Big Endian)** using `htons` and `htonl`.
- Listing 17–3 builds the multi-threaded listening server. Each accepted
  connection is dispatched to an isolated `std::jthread`, while use of
  `std::stop_token` allows for graceful server shutdown.
- Listing 17–4 creates a network client designed to run in the main
  thread, which connects to the protocol server.
- Listing 17–5 simulates a distributed Client-Server database engine:
  1.  The **Database Server** boots, populates an in-memory table, and
      listens on port `8080`.
  2.  The **Client** connects over a local loopback TCP socket,
      serialises raw SQL query strings inside typed binary frames
      (`MessageType::ExecuteQuery`), and transmits them over the wire.
  3.  The server receives the frame, extracts the SQL text, parses and
      executes the relational query plan, formats the result into a
      tabular payload, and returns the result in a
      `MessageType::QueryResult` frame (as in Figure 17–7).

<figure id="fig:db-connect">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NTUuMiIgaGVp
Z2h0PSIyODQuODAwMDIiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNjU1LjIgMjg0Ljgw
MDAyIj4KICA8c3R5bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xl
LCAuc3ZnYm9iIHJlY3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJv
a2Utd2lkdGg6IDEuNTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAg
c3Ryb2tlLWxpbmVjYXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdi
b2IgdGV4dCB7CiAgd2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWls
eTogQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6
IDEzcHg7Cn0KCi5zdmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6
IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7
Cn0KCi5zdmdib2IgLmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxl
ZCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5u
b2ZpbGwgewogIGZpbGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93
IHsKICBtYXJrZXItZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fy
cm93IHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9kaWFtb25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUp
Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwo
I29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBt
YXJrZXItc3RhcnQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Jp
Z19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVy
bCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9
ImFycm93IiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRo
PSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAg
PHBvbHlnb24gcG9pbnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2Vy
PgogICAgPG1hcmtlciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiBy
ZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+
PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjIiIGNsYXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJvcGVuX2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+
PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtl
cj4KICA8L2RlZnM+CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9
IjY1NS4yIiBoZWlnaHQ9IjI4NC44MDAwMiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjI1LjM1IiB5PSIyNy4zMDAwMDEiID5DbGllbnQ8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzkuOTUwMDA1IiB5PSIyNy4zMDAwMDEiID5Ob2RlPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQzMC45NSIgeT0iMjcuMzAwMDAxIiA+
U2VydmVyPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ4NS41NTAwMiIg
eT0iMjcuMzAwMDAxIiA+Tm9kZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI4Ny43NSIgeT0iODkuNzAwMDA1IiA+MS48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMTExLjE1IiB5PSI4OS43MDAwMDUiID5UQ1A8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMTQyLjM1IiB5PSI4OS43MDAwMDUiID5Db25uZWN0PC90ZXh0Pgog
IDxwb2x5Z29uIHBvaW50cz0iNDYwLjIsOTcuNSA0NjgsMTAxLjQgNDYwLjIsMTA1LjMiIGNsYXNz
PSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI4Ny43
NSIgeT0iMTM2LjUiID4yLjwvdGV4dD4KICA8cG9seWdvbiBwb2ludHM9IjQ2MC4yLDE0NC4zIDQ2
OCwxNDguMiA0NjAuMiwxNTIuMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9Ijg3Ljc1IiB5PSIyMzAuMSIgPjMuPC90ZXh0PgogIDxwb2x5
Z29uIHBvaW50cz0iODUuOCwyMzcuOTAwMDEgNzgsMjQxLjggODUuOCwyNDUuNzAwMDEiIGNsYXNz
PSJmaWxsZWQiPjwvcG9seWdvbj4KICA8cG9seWdvbiBwb2ludHM9IjQ5MS40MDAwMiwyMDYuNzAw
MDEgNDgzLjYsMjEwLjYgNDkxLjQwMDAyLDIxNC41IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTI0LjU1IiB5PSIxNjcuNyIgPkV2YWx1
YXRlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU5NC43NSIgeT0iMTY3
LjciID5BU1Q8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTI0LjU1IiB5
PSIxODMuMyIgPkxvZ2ljYWw8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NTg2Ljk1IiB5PSIxODMuMyIgPlNjYW48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iNTI0LjU1IiB5PSIxOTguOTAwMDEiID5Qcm9qZWN0PC90ZXh0PgogIDxwYXRoIGQ9Ik0g
NTkyLjgsMTg3LjIwMDAxIEEgMTUuNiwxNS42IDAsMCwwIDU5Mi44LDIwMi44IiBjbGFzcz0ibm9m
aWxsIj48L3BhdGg+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTk0Ljc1IiB5PSIx
OTguOTAwMDEiID7PgDwvdGV4dD4KICA8cGF0aCBkPSJNIDYwMC42MDAwNCwxODcuMjAwMDEgQSAx
NS42LDE1LjYgMCwwLDEgNjAwLjYwMDA0LDIwMi44IiBjbGFzcz0ibm9maWxsIj48L3BhdGg+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTI0LjU1IiB5PSIyMTQuNSIgPkZvcm1hdDwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1NzkuMTUiIHk9IjIxNC41IiA+
UmVzdWx0PC90ZXh0PgogIDxwYXRoIGQ9Ik0gNjMxLjgsMTcxLjYgQSAxNS42LDE1LjYgMCwwLDAg
NjMxLjgsMTg3LjIwMDAxIiBjbGFzcz0ibm9maWxsIj48L3BhdGg+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNjMzLjc1IiB5PSIxODMuMyIgPs+DPC90ZXh0PgogIDxwYXRoIGQ9Ik0g
NjM5LjYwMDA0LDE3MS42IEEgMTUuNiwxNS42IDAsMCwxIDYzOS42MDAwNCwxODcuMjAwMDEiIGNs
YXNzPSJub2ZpbGwiPjwvcGF0aD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1
IiB5PSI0Mi45IiA+KFRlcm1pbmFsIFVJKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI0MTUuMzUiIHk9IjQyLjkiID4oQ29yZURCIEhvc3QpPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjIwNC43NSIgeT0iODkuNzAwMDA1IiA+KEhhbmRzaGFrZSk8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTExLjE1IiB5PSIxMzYuNSIg
PkZyYW1lOiBbRVhFQ19TUUxdICYjMzk7U0VMRUNUIG5hbWUgRlJPTSB1c2VyczsmIzM5OzwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMTEuMTUiIHk9IjIzMC4xIiA+RnJh
bWU6IFtRVUVSWV9SRVNVTFRdICYjMzk7QWRhIExvdmVsYWNlIHwgOTIwMCYjMzk7PC90ZXh0Pgog
IDxnPgogICAgPGxpbmUgeDE9IjMuOSIgeTE9IjcuOCIgeDI9IjEyOC43IiB5Mj0iNy44IiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIzLjkiIHkxPSI3LjgiIHgyPSIzLjkiIHky
PSI1NC42MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjEyOC43IiB5
MT0iNy44IiB4Mj0iMTI4LjciIHkyPSI1NC42MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
ICAgPGxpbmUgeDE9IjMuOSIgeTE9IjU0LjYwMDAwMiIgeDI9IjEyOC43IiB5Mj0iNTQuNjAwMDAy
IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSI2Ni4zIiB5MT0iNTQuNjAwMDAy
IiB4Mj0iNjYuMyIgeTI9IjI2NS4yIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgx
PSI2Ni4zIiB5MT0iMTAxLjQiIHgyPSI0NjAuMiIgeTI9IjEwMS40IiBjbGFzcz0ic29saWQiPjwv
bGluZT4KICAgIDxsaW5lIHgxPSI2Ni4zIiB5MT0iMTQ4LjIiIHgyPSI0NjAuMiIgeTI9IjE0OC4y
IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8L2c+CiAgPGc+CiAgICA8bGluZSB4MT0iNDA5LjUi
IHkxPSI3LjgiIHgyPSI1MzQuMyIgeTI9IjcuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8
bGluZSB4MT0iNDA5LjUiIHkxPSI3LjgiIHgyPSI0MDkuNSIgeTI9IjU0LjYwMDAwMiIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iNTM0LjMiIHkxPSI3LjgiIHgyPSI1MzQuMyIg
eTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iNDA5LjUi
IHkxPSI1NC42MDAwMDIiIHgyPSI1MzQuMyIgeTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgICA8bGluZSB4MT0iNDcxLjkwMDAyIiB5MT0iNTQuNjAwMDAyIiB4Mj0iNDcxLjkw
MDAyIiB5Mj0iMjY1LjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9Ijg1Ljgi
IHkxPSIyNDEuOCIgeDI9IjQ3MS45MDAwMiIgeTI9IjI0MS44IiBjbGFzcz0ic29saWQiPjwvbGlu
ZT4KICA8L2c+CiAgPGc+CiAgICA8bGluZSB4MT0iNDgzLjYiIHkxPSIxNjMuOCIgeDI9IjUxMC45
MDAwMiIgeTI9IjE2My44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSI1MTAu
OTAwMDIiIHkxPSIxNjMuOCIgeDI9IjUxMC45MDAwMiIgeTI9IjIxMC42IiBjbGFzcz0ic29saWQi
PjwvbGluZT4KICAgIDxsaW5lIHgxPSI0OTEuNDAwMDIiIHkxPSIyMTAuNiIgeDI9IjUxMC45MDAw
MiIgeTI9IjIxMC42IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 17–7: Time-Based Diagram of Querying a Remote
Database</figcaption>
</figure>

Listing 17–1: RAII Socket Wrappers and Framing (SocketRAII.hpp):

In [ ]:
#ifndef SOCKET_RAII_HPP
#define SOCKET_RAII_HPP

#include <iostream>
#include <string>
#include <vector>
#include <memory>
#include <stdexcept>
#include <span>
#include <cstdint>
#include <cstring>

// POSIX Networking Headers
#include <sys/types.h>
#include <sys/socket.h>
#include <netinet/in.h>
#include <arpa/inet.h>
#include <unistd.h>
#include <fcntl.h>

namespace ModernNet {

// RAII Wrapper around a POSIX file descriptor
class Socket {
public:
    Socket() : m_fd(-1) {}
    explicit Socket(int fd) : m_fd(fd) {}

    ~Socket() {
        close_socket();
    }

    // Move-only semantics (no accidental descriptor duplication)
    Socket(const Socket&) = delete;
    Socket& operator=(const Socket&) = delete;

    Socket(Socket&& other) noexcept : m_fd(other.m_fd) {
        other.m_fd = -1;
    }

    Socket& operator=(Socket&& other) noexcept {
        if (this != &other) {
            close_socket();
            m_fd = other.m_fd;
            other.m_fd = -1;
        }
        return *this;
    }

    [[nodiscard]] int native_handle() const noexcept { return m_fd; }
    [[nodiscard]] bool is_valid() const noexcept { return m_fd >= 0; }

    void shutdown(int how = SHUT_RDWR) noexcept {
        if (m_fd >= 0) {
            ::shutdown(m_fd, how);
        }
    }

    void close_socket() noexcept {
        if (m_fd >= 0) {
            ::close(m_fd);
            m_fd = -1;
        }
    }

    // Guaranteed complete send over stream
    void send_all(std::span<const std::byte> data) const {
        size_t total_sent = 0;
        const auto* raw_ptr = reinterpret_cast<const char*>(data.data());

        while (total_sent < data.size()) {
            ssize_t bytes_sent = ::send(m_fd, raw_ptr + total_sent, data.size() - total_sent, 0);
            if (bytes_sent <= 0) {
                throw std::runtime_error("Socket write failure: " + std::string(strerror(errno)));
            }
            total_sent += static_cast<size_t>(bytes_sent);
        }
    }

    // Guaranteed complete receive over stream
    void recv_exact(std::span<std::byte> destination) const {
        size_t total_read = 0;
        auto* raw_ptr = reinterpret_cast<char*>(destination.data());

        while (total_read < destination.size()) {
            ssize_t bytes_read = ::recv(m_fd, raw_ptr + total_read, destination.size() - total_read, 0);
            if (bytes_read == 0) {
                throw std::runtime_error("Connection closed by peer (EOF).");
            }
            if (bytes_read < 0) {
                throw std::runtime_error("Socket read failure: " + std::string(strerror(errno)));
            }
            total_read += static_cast<size_t>(bytes_read);
        }
    }

private:
    int m_fd;
};

} // namespace ModernNet

#endif // SOCKET_RAII_HPP

Listing 17–2: Frame Layout and Network Endianness (Protocol.hpp):

In [ ]:
#ifndef PROTOCOL_HPP
#define PROTOCOL_HPP

//#include "SocketRAII.hpp"
#include <string_view>
#include <bit>

namespace ModernNet {

enum class MessageType : uint16_t {
    Ping         = 0x0001,
    Pong         = 0x0002,
    ExecuteQuery = 0x0010,
    QueryResult  = 0x0011,
    Error        = 0x00FF
};

#pragma pack(push, 1)
struct FrameHeader {
    uint16_t magic{0xCAFE};       // Protocol Identifier
    uint16_t msg_type{0};         // MessageType
    uint32_t payload_length{0};   // Size of payload in bytes
};
#pragma pack(pop)

static_assert(sizeof(FrameHeader) == 8, "FrameHeader must be exactly 8 bytes packed.");

struct Message {
    MessageType type;
    std::string payload;
};

class ProtocolEngine {
public:
    static constexpr uint16_t PROTOCOL_MAGIC = 0xCAFE;

    // Serialise and transmit a message frame across the socket
    static void send_message(const Socket& sock, MessageType type, std::string_view payload) {
        FrameHeader header;
        header.magic = htons(PROTOCOL_MAGIC);
        header.msg_type = htons(static_cast<uint16_t>(type));
        header.payload_length = htonl(static_cast<uint32_t>(payload.size()));

        // Transmit header
        std::span<const std::byte> header_bytes(
            reinterpret_cast<const std::byte*>(&header), sizeof(FrameHeader)
        );
        sock.send_all(header_bytes);

        // Transmit payload
        if (!payload.empty()) {
            std::span<const std::byte> payload_bytes(
                reinterpret_cast<const std::byte*>(payload.data()), payload.size()
            );
            sock.send_all(payload_bytes);
        }
    }

    // Block and reconstruct an incoming framed message
    static Message receive_message(const Socket& sock) {
        FrameHeader raw_header;
        std::span<std::byte> header_dest(
            reinterpret_cast<std::byte*>(&raw_header), sizeof(FrameHeader)
        );
        sock.recv_exact(header_dest);

        // Decode network byte order
        uint16_t magic = ntohs(raw_header.magic);
        uint16_t type_val = ntohs(raw_header.msg_type);
        uint32_t length = ntohl(raw_header.payload_length);

        if (magic != PROTOCOL_MAGIC) {
            throw std::runtime_error("Protocol Error: Invalid header magic bytes 0x" + 
                                     std::to_string(magic));
        }

        // Allocate payload buffer and read exact payload length
        std::string payload;
        payload.resize(length);

        if (length > 0) {
            std::span<std::byte> payload_dest(
                reinterpret_cast<std::byte*>(payload.data()), length
            );
            sock.recv_exact(payload_dest);
        }

        return Message{
            .type = static_cast<MessageType>(type_val),
            .payload = std::move(payload)
        };
    }
};

} // namespace ModernNet

#endif // PROTOCOL_HPP

Listing 17–3: Multi-Threaded TCP Server (Server.hpp):

In [ ]:
#ifndef SERVER_HPP
#define SERVER_HPP

//#include "SocketRAII.hpp"
//#include "Protocol.hpp"
#include <thread>
#include <vector>
#include <atomic>
#include <functional>
#include <mutex>

namespace ModernNet {

class TcpServer {
public:
    using MessageHandler = std::function<void(const Socket& client_sock, const Message& msg)>;

    explicit TcpServer(uint16_t port) : m_port(port), m_is_running(false) {}

    ~TcpServer() {
        stop();
    }

    void start(MessageHandler handler) {
        m_handler = std::move(handler);
        
        int raw_fd = ::socket(AF_INET, SOCK_STREAM, 0);
        if (raw_fd < 0) {
            throw std::runtime_error("Failed to create server socket: " + std::string(strerror(errno)));
        }
        m_listen_socket = Socket(raw_fd);

        // Set SO_REUSEADDR to prevent "Address already in use" binding errors during restarts
        int opt = 1;
        if (::setsockopt(m_listen_socket.native_handle(), SOL_SOCKET, SO_REUSEADDR, &opt, sizeof(opt)) < 0) {
            throw std::runtime_error("setsockopt(SO_REUSEADDR) failed.");
        }

        sockaddr_in server_addr{};
        server_addr.sin_family = AF_INET;
        server_addr.sin_addr.s_addr = INADDR_ANY; // Bind to all interfaces
        server_addr.sin_port = htons(m_port);

        if (::bind(m_listen_socket.native_handle(), reinterpret_cast<sockaddr*>(&server_addr), sizeof(server_addr)) < 0) {
            throw std::runtime_error("Bind failed on port " + std::to_string(m_port) + ": " + strerror(errno));
        }

        if (::listen(m_listen_socket.native_handle(), SOMAXCONN) < 0) {
            throw std::runtime_error("Listen failed: " + std::string(strerror(errno)));
        }

        m_is_running = true;
        std::cout << "[Server] Listening on 0.0.0.0:" << m_port << "...\n";

        // Spawn accept loop thread
        m_accept_thread = std::jthread([this](std::stop_token st) {
            accept_loop(st);
        });
    }

    void stop() {
        if (!m_is_running.exchange(false)) return;

        std::cout << "[Server] Shutting down...\n";
        m_listen_socket.shutdown();
        m_listen_socket.close_socket();

        if (m_accept_thread.joinable()) {
            m_accept_thread.request_stop();
        }

        // Join active client threads
        std::lock_guard<std::mutex> lock(m_threads_mutex);
        m_worker_threads.clear();
    }

private:
    uint16_t m_port;
    Socket m_listen_socket;
    std::atomic<bool> m_is_running{false};
    MessageHandler m_handler;
    std::jthread m_accept_thread;

    std::mutex m_threads_mutex;
    std::vector<std::jthread> m_worker_threads;

    void accept_loop(std::stop_token st) {
        while (!st.stop_requested() && m_is_running) {
            sockaddr_in client_addr{};
            socklen_t client_len = sizeof(client_addr);

            int client_fd = ::accept(m_listen_socket.native_handle(), 
                                     reinterpret_cast<sockaddr*>(&client_addr), 
                                     &client_len);

            if (client_fd < 0) {
                if (!m_is_running) break; // Normal socket shutdown
                std::cerr << "[Server] Accept error: " << strerror(errno) << "\n";
                continue;
            }

            char ip_str[INET_ADDRSTRLEN];
            inet_ntop(AF_INET, &(client_addr.sin_addr), ip_str, INET_ADDRSTRLEN);
            std::cout << "[Server] Accepted connection from " << ip_str << ":" 
                      << ntohs(client_addr.sin_port) << "\n";

            Socket client_sock(client_fd);

            // Dispatch to worker thread
            std::lock_guard<std::mutex> lock(m_threads_mutex);
            m_worker_threads.emplace_back([this, sock = std::move(client_sock)](std::stop_token) mutable {
                handle_client(std::move(sock));
            });
        }
    }

    void handle_client(Socket sock) {
        try {
            while (m_is_running) {
                Message msg = ProtocolEngine::receive_message(sock);
                m_handler(sock, msg);
            }
        } catch (const std::exception& e) {
            // Client disconnect or socket error
            // (Standard EOF terminates loop cleanly)
        }
    }
};

} // namespace ModernNet

#endif // SERVER_HPP

Listing 17–4: Network Client (Client.hpp):

In [ ]:
#ifndef CLIENT_HPP
#define CLIENT_HPP

//#include "SocketRAII.hpp"
//#include "Protocol.hpp"

namespace ModernNet {

class TcpClient {
public:
    TcpClient() = default;

    void connect_to(std::string_view host, uint16_t port) {
        int raw_fd = ::socket(AF_INET, SOCK_STREAM, 0);
        if (raw_fd < 0) {
            throw std::runtime_error("Failed to create client socket: " + std::string(strerror(errno)));
        }
        m_socket = Socket(raw_fd);

        sockaddr_in server_addr{};
        server_addr.sin_family = AF_INET;
        server_addr.sin_port = htons(port);

        if (::inet_pton(AF_INET, host.data(), &server_addr.sin_addr) <= 0) {
            throw std::runtime_error("Invalid address / Address not supported: " + std::string(host));
        }

        if (::connect(m_socket.native_handle(), reinterpret_cast<sockaddr*>(&server_addr), sizeof(server_addr)) < 0) {
            throw std::runtime_error("Connection to " + std::string(host) + ":" + 
                                     std::to_string(port) + " failed: " + strerror(errno));
        }

        std::cout << "[Client] Successfully connected to " << host << ":" << port << "\n";
    }

    void send_request(MessageType type, std::string_view payload) {
        ProtocolEngine::send_message(m_socket, type, payload);
    }

    Message receive_response() {
        return ProtocolEngine::receive_message(m_socket);
    }

    void disconnect() {
        m_socket.close_socket();
    }

private:
    Socket m_socket;
};

} // namespace ModernNet

#endif // CLIENT_HPP

Listing 17–5: Complete Network Demonstration Driver:

In [ ]:
//#include "SocketRAII.hpp"
//#include "Protocol.hpp"
//#include "Server.hpp"
//#include "Client.hpp"

#include <iostream>
#include <sstream>
#include <chrono>

int main() {
    using namespace ModernNet;

    const uint16_t SERVER_PORT = 8080;

    std::cout << "====================================================\n";
    std::cout << "     Modern C++ Network Socket Framework Demo       \n";
    std::cout << "====================================================\n\n";

    // 1. Initialise and Start the Multi-threaded Server
    TcpServer server(SERVER_PORT);

    server.start([](const Socket& client_sock, const Message& msg) {
        switch (msg.type) {
            case MessageType::Ping: {
                std::cout << "[Server] Received PING. Responding with PONG...\n";
                ProtocolEngine::send_message(client_sock, MessageType::Pong, "PONG_ACK");
                break;
            }
            case MessageType::ExecuteQuery: {
                std::cout << "[Server] Received SQL Request: " << msg.payload << "\n";
                
                // Simulated Database Response
                std::string response_data;
                if (msg.payload.find("SELECT") != std::string::npos) {
                    response_data = 
                        "+-----+-------------------+------------+\n"
                        "| ID  | NAME              | SALARY     |\n"
                        "+-----+-------------------+------------+\n"
                        "| 101 | Ada Lovelace      | 8500.50    |\n"
                        "| 102 | Alan Turing       | 9200.00    |\n"
                        "+-----+-------------------+------------+\n"
                        "(2 rows returned across network stream)";
                } else {
                    response_data = "OK: Statement executed successfully.";
                }

                ProtocolEngine::send_message(client_sock, MessageType::QueryResult, response_data);
                break;
            }
            default: {
                std::cerr << "[Server] Unknown message type received.\n";
                ProtocolEngine::send_message(client_sock, MessageType::Error, "UNKNOWN_COMMAND");
                break;
            }
        }
    });

    // Give server thread a moment to bind and enter listen state
    std::this_thread::sleep_for(std::chrono::milliseconds(50));

    // 2. Initialise Client and Connect
    try {
        TcpClient client;
        client.connect_to("127.0.0.1", SERVER_PORT);

        // Test 1: Send Ping Frame
        std::cout << "\n[Client] Sending PING frame...\n";
        client.send_request(MessageType::Ping, "");
        Message ping_resp = client.receive_response();
        std::cout << "[Client] Server reply -> Type: " 
                  << (ping_resp.type == MessageType::Pong ? "PONG" : "OTHER") 
                  << ", Payload: " << ping_resp.payload << "\n";

        // Test 2: Send Networked SQL Query Frame
        std::string sql_query = "SELECT id, name, salary FROM engineers WHERE salary > 8000.0;";
        std::cout << "\n[Client] Transmitting SQL Query: " << sql_query << "\n";
        client.send_request(MessageType::ExecuteQuery, sql_query);

        Message query_resp = client.receive_response();
        std::cout << "[Client] Query Response Received:\n\n" << query_resp.payload << "\n\n";

        client.disconnect();
    } catch (const std::exception& e) {
        std::cerr << "[Client Error]: " << e.what() << "\n";
    }

    // 3. Graceful Server Shutdown
    server.stop();
    std::cout << "[System] Network demonstration finished cleanly.\n";

    return 0;
}

In [ ]:
main(); // Invocation of main program for Jupyter notebook running the xeus-cpp kernel

### 17.9 Summary

In this chapter, we have moved past the illusion of the single machine
and examined the mechanics of networked systems:

- **The Fallacies of Distributed Computing** are a reminder that
  networks are inherently asynchronous, band-limited, and prone to
  partial failure.
- **The TCP/IP Layered Architecture** isolates concerns across Physical,
  Link, Internet, Transport, and Application domains via systematic
  header encapsulation.
- **TCP Guarantees Reliability** over an unreliable IP backbone through
  sequence numbers, cumulative acknowledgements, sliding windows, and
  connection handshakes.
- **TCP is a Stream, Not a Message Bus.** Applications must explicitly
  manage framing using techniques like Length-Prefixed Headers (TLV) to
  avoid packet coalescing and fragmentation bugs.
- **Modern C++20 Sockets** leverage RAII to prevent descriptor leaks,
  enforce network byte order for cross-platform portability, and use
  `std::jthread` for clean concurrent connection management.

### 17.10 Programming Challenges

#### Exercise 17.1: Non-Blocking Sockets and Asynchronous Polling (`poll`)

**Context and Theory:**

Allocating a dedicated operating system thread per client connection
(`thread-per-connection`) collapses under high concurrency ($10^4$ to
$10^6$ active connections, known historically as the **C10K problem**).
Thread stacks consume physical virtual memory (typically $1\text{ MB}$
to $8\text{ MB}$ per thread), and excessive thread counts degrade system
performance through CPU context-switching overhead.

High-performance networked servers employ **I/O Multiplexing** via
non-blocking sockets (see Figure 17–8). By setting the `O_NONBLOCK` file
status flag:

``` cpp
int flags = fcntl(fd, F_GETFL, 0);
fcntl(fd, F_SETFL, flags | O_NONBLOCK);
```

Calls to `recv()` or `accept()` return immediately with an error code
`EWOULDBLOCK` or `EAGAIN` if the kernel buffers are empty, rather than
suspending the calling thread. The application monitors thousands of
descriptors using a single thread via the POSIX `poll()` (or Linux
`epoll`) system interface, waking up only when the operating system
kernel notifies it of readiness events (`POLLIN`, `POLLOUT`, `POLLERR`).

<figure id="fig:non-blocking-sokets">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI3MDIiIGhlaWdo
dD0iODIiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNzAyIDgyIj4KICA8c3R5bGU+LnN2
Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xlLCAuc3ZnYm9iIHJlY3QsIC5z
dmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJva2Utd2lkdGg6IDEuNTsKICBz
dHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAgc3Ryb2tlLWxpbmVjYXA6IHJv
dW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdib2IgdGV4dCB7CiAgd2hpdGUt
c3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWlseTogQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDEzcHg7Cn0KCi5zdmdib2Ig
cmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRyYW5zcGFyZW50Owp9Cgou
c3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0KCi5zdmdib2IgLmZpbGxl
ZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7CiAgZmlsbDogdHJhbnNw
YXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2ZpbGwgewogIGZpbGw6IHRy
YW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsKICBtYXJrZXItZW5kOiB1
cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93IHsKICBtYXJrZXItc3Rh
cnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtl
ci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9kaWFtb25kIHsK
ICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtl
ZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRf
bWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29wZW5fY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAg
bWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYmlnX29wZW5fY2lyY2xl
KTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFycm93IiB2aWV3Qm94PSIt
MiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIw
LDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0i
ZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9wb2x5Z29uPgogICAgPC9t
YXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQi
IHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1z
dGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJm
aWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJvcGVuX2NpcmNs
ZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1h
cmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xl
IGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21h
cmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMyIg
Y2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+CiAgPHJl
Y3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjcwMiIgaGVpZ2h0PSI4MiI+
PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9IjExLjcwMDAw
MSIgPktlcm5lbDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1IiB5
PSIyNy4zMDAwMDEiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijku
NzUiIHk9IjQyLjkiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijku
NzUiIHk9IjU4LjUiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjY0
LjM1IiB5PSIxMS43MDAwMDEiID5Tb2NrZXQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMTE4Ljk1MDAwNSIgeT0iMTEuNzAwMDAxIiA+QnVmZmVyczwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMTIuNTUiIHk9IjI3LjMwMDAwMSIgPl08L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjEyLjU1IiB5PSI0Mi45IiA+XTwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMTIuNTUiIHk9IjU4LjUiID5dPC90
ZXh0PgogIDxwb2x5Z29uIHBvaW50cz0iMjk2LjQsMzUuMTAwMDAyIDMwNC4yLDM5IDI5Ni40LDQy
LjkiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIzMTMuOTUiIHk9IjQyLjkiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjU0Ny45NSIgeT0iNDIuOSIgPl08L3RleHQ+CiAgPGxpbmUgeDE9IjU2MS42MDAwNCIg
eTE9IjM5IiB4Mj0iNTc3LjIiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHln
b24gcG9pbnRzPSI1NzcuMiwzNS4xMDAwMDIgNTg1LDM5IDU3Ny4yLDQyLjkiIGNsYXNzPSJmaWxs
ZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iNTYxLjYwMDA0IiB5MT0iNTQuNjAwMDAyIiB4Mj0i
NTc3LjIiIHkyPSI1NC42MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBv
aW50cz0iNTc3LjIsNTAuNyA1ODUsNTQuNjAwMDAyIDU3Ny4yLDU4LjUiIGNsYXNzPSJmaWxsZWQi
PjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjEuNzUiIHk9IjEx
LjcwMDAwMSIgPnBvbGwoKSBFdmVudCBMb29wPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjI1LjM1IiB5PSIyNy4zMDAwMDEiID5Tb2NrZXQgMTAxOiBEYXRhIFJlYWR5PC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1LjM1IiB5PSI0Mi45IiA+U29j
a2V0IDEwMjogSWRsZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjku
NTUwMDIiIHk9IjQyLjkiID5wb2xsKCkgd2FrZXMgc2luZ2xlIHRocmVhZDwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1OTQuNzUiIHk9IjQyLjkiID5EaXNwYXRjaCAjMTAx
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1LjM1IiB5PSI1OC41IiA+
U29ja2V0IDEwMzogRGF0YSBSZWFkeTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSI1OTQuNzUiIHk9IjU4LjUiID5EaXNwYXRjaCAjMTAzPC90ZXh0PgogIDxnPgogICAgPGxp
bmUgeDE9IjIyNi4yMDAwMSIgeTE9IjIzLjQwMDAwMiIgeDI9IjI1My41IiB5Mj0iMjMuNDAwMDAy
IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIyNTMuNSIgeTE9IjIzLjQwMDAw
MiIgeDI9IjI1My41IiB5Mj0iNTQuNjAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxs
aW5lIHgxPSIyNTMuNSIgeTE9IjM5IiB4Mj0iMjk2LjQiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgICA8bGluZSB4MT0iMjI2LjIwMDAxIiB5MT0iNTQuNjAwMDAyIiB4Mj0iMjUzLjUi
IHkyPSI1NC42MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 17–8: Performance Networking with Non-Blocking
Sockets</figcaption>
</figure>

**Specification:**

Implement an asynchronous, single-threaded `EventLoop` server class
using non-blocking sockets and POSIX `poll()`. The server must manage
active client connections in a dynamic array of `pollfd` structures,
handle partial reads without blocking, and gracefully prune disconnected
descriptors.

``` cpp
class EventLoopServer {
public:
    explicit EventLoopServer(uint16_t port);
    void run(std::stop_token stop_token);
    void register_callback(std::function<void(int client_fd, std::span<const std::byte>)> on_data);
private:
    Socket m_listen_sock;
    std::vector<pollfd> m_poll_fds;
    std::unordered_map<int, Socket> m_client_sockets;
    std::function<void(int, std::span<const std::byte>)> m_on_data;

    void set_non_blocking(int fd);
    void accept_pending_connections();
    void handle_client_io(size_t index);
};
```

**Verification Tasks:**

1.  **Non-Blocking Invariant:** Connect $50$ concurrent client sockets
    that remain completely silent. Verify that `EventLoopServer::run`
    consumes zero excessive CPU time ($< 1\%$ CPU load), successfully
    idling inside the `poll()` kernel wait.
2.  **Multiplexed Read Verification:** Transmit $100$ distinct message
    payloads simultaneously from $10$ client threads into the
    single-threaded server. Verify that all $100$ payloads are
    completely received and dispatched without frame corruption or
    missing bytes.
3.  **Disconnection Cleanup:** Disconnect $5$ clients abruptly (closing
    their endpoints). Verify that the server detects `POLLHUP` or
    zero-byte reads, closes the sockets, and compacts the `m_poll_fds`
    array to size $45$.

#### Exercise 17.2: Sliding-Window Flow Control over UDP

**Context and Theory:**

TCP guarantees that a fast sender does not overrun the memory buffers of
a slow receiver through **Sliding-Window Flow Control**. In contrast,
UDP offers no flow control; datagrams arriving at a congested kernel
buffer are dropped without warning.

The flow control window represents the permissible range of sequence
numbers the sender may transmit without waiting for an acknowledgement:

$$\text{Usable Window} = [S_{\text{UNA}}, S_{\text{UNA}} + W - 1]$$

Where:

- $S_{\text{UNA}}$ is the oldest unacknowledged sequence number.
- $W$ is the receiver's advertised buffer window size (in bytes or
  packets).

As the receiver processes datagrams from its queue, it transmits an
acknowledgement (ACK) containing the updated window size, sliding the
transmission window forward across the byte stream (see Figure 17–9).

<figure id="fig:sender-window">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NDYiIGhlaWdo
dD0iNTAuODAwMDAzIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDU0NiA1MC44MDAwMDMi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNw
eDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJh
bnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoK
LnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmls
bCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlh
bW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFy
a2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0K
Ci5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Bl
bl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtl
ci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29w
ZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNi
aWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJy
b3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9Ijci
IG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9s
eWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAg
ICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9
IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1y
ZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3Bv
bHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2Np
cmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3
Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQi
IGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgog
IDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTQ2
IiBoZWlnaHQ9IjUwLjgwMDAwMyI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iNy44LDE5LjUg
MCwyMy40MDAwMDIgNy44LDI3LjMwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxs
aW5lIHgxPSI3LjgiIHkxPSIyMy40MDAwMDIiIHgyPSI2Mi40IiB5Mj0iMjMuNDAwMDAyIiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjYyLjQsMTkuNSA3MC4yMDAwMDUs
MjMuNDAwMDAyIDYyLjQsMjcuMzAwMDAxIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHBv
bHlnb24gcG9pbnRzPSI4NS44LDE5LjUgNzgsMjMuNDAwMDAyIDg1LjgsMjcuMzAwMDAxIiBjbGFz
cz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9Ijg1LjgiIHkxPSIyMy40MDAwMDIiIHgy
PSIyNjUuMiIgeTI9IjIzLjQwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjc0Ljk1IiB5PSIyNy4zMDAwMDEiID5XPC90ZXh0PgogIDxs
aW5lIHgxPSIyODguNiIgeTE9IjIzLjQwMDAwMiIgeDI9IjQzNi44MDAwMiIgeTI9IjIzLjQwMDAw
MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI0MzYuODAwMDIsMTku
NSA0NDQuNiwyMy40MDAwMDIgNDM2LjgwMDAyLDI3LjMwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9w
b2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjExLjcwMDAw
MSIgPlsgQUNLZWQgXSBbIFNlbnQsIFVuQUNLZWQgKFNfVU5BKSBdIFsgQWxsb3dlZCwgTm90IFNl
bnQgXSBbIEJsb2NrZWQgXTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 17–9: Sender Window</figcaption>
</figure>

**Specification:**

Construct a simulated `SlidingWindowSender` and `SlidingWindowReceiver`
over raw UDP sockets (`SOCK_DGRAM`). Implement cumulative
acknowledgements, an adjustable window size $W$, and a retransmission
timeout (RTO) calculated using Jacobson's algorithm for Smoothed Round
Trip Time (SRTT):

$$\text{SRTT} \leftarrow (1 - \alpha) \cdot \text{SRTT} + \alpha \cdot \text{RTT}_{\text{sample}} \qquad (\alpha = 0.125)$$

``` cpp
struct PacketHeader {
    uint32_t seq_num;
    uint32_t ack_num;
    uint16_t window_size;
};

class SlidingWindowSender {
public:
    SlidingWindowSender(Socket sock, sockaddr_in dest, uint16_t initial_w);
    void send_packet(std::span<const std::byte> payload);
    void on_ack_received(uint32_t ack, uint16_t advertised_window);
    void check_timeouts_and_retransmit();
private:
    uint32_t m_next_seq_num{1};
    uint32_t m_send_unacked{1};
    uint16_t m_window_size;
    std::map<uint32_t, std::vector<std::byte>> m_unacked_packets;
};
```

**Verification Tasks:**

1.  **Window Exhaustion Stalling:** Set window capacity $W = 3$ packets.
    Attempt to send $5$ packets without receiving ACKs. Verify that
    packets $4$ and $5$ are blocked from physical network transmission
    until packet $1$ is acknowledged.
2.  **Cumulative ACK Advancement:** Transmit packets $1, 2, 3$. Simulate
    receiving an ACK with `ack_num = 4`. Verify that packets $1$, $2$,
    and $3$ are simultaneously retired from the retransmission buffer.
3.  **Loss Recovery and Resend:** Drop packet $2$ over an artificial
    loss proxy. Verify that upon RTO expiration, packet $2$ is
    retransmitted automatically and the receiver reassembles the stream
    in strict sequential order.

#### Exercise 17.3: Robust Length-Prefixed Framing with CRC-32 Validation

**Context and Theory:**

Physical network transmission can suffer from hardware-level data
corruption (e.g., bit-flips caused by electromagnetic interference or
faulty router memory). While Ethernet and TCP include 16-bit checksums,
their error detection capabilities are mathematically weak against burst
errors.

The **Cyclic Redundancy Check (CRC-32)** treats a message of $k$ bits as
polynomial coefficients over the Galois Field $\text{GF}(2)$, computing
the remainder after division by a standard generator polynomial:

$$G(x) = x^{32} + x^{26} + x^{23} + \dots + x^2 + x + 1 \quad (\text{IEEE 802.3 } \mathtt{0xEDB88320})$$

An application protocol combines **Length-Prefixed Framing** with CRC-32
validation. If the computed CRC of the received payload does not match
the header's declared CRC, the frame is marked as corrupted and dropped
before reaching application logic.

<figure id="fig:crc-validation">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MjIuNjAwMDQi
IGhlaWdodD0iMzUuMiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1MjIuNjAwMDQgMzUu
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IENvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAx
M3B4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0
cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9
Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQg
ewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9m
aWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9k
aWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3Zn
Ym9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNv
cGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFy
a2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdf
b3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJh
cnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxw
b2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVm
WT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0
LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwv
cG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwv
Y2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZp
ZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0i
NCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+
CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1
MjIuNjAwMDQiIGhlaWdodD0iMzUuMiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjEuOTUiIHk9IjExLjcwMDAwMSIgPlsgTWFnaWM6IDJCIF1bIExlbmd0aDogNEIgXVsg
Q1JDLTMyOiA0QiBdWyBQYXlsb2FkIERhdGE6IE4gQnl0ZXMgXTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 17–10: Binary Frame</figcaption>
</figure>

**Specification:**

Extend the `ProtocolEngine` from Chapter 17 to incorporate CRC-32
integrity validation. Implement a streaming `FrameDecoder` class that
handles fragmented packet arrivals across `recv()` calls, accumulating
bytes until a full frame is verified.

``` cpp
class FrameDecoder {
public:
    void feed_bytes(std::span<const std::byte> incoming);
    std::optional<Message> extract_next_frame();
private:
    std::vector<std::byte> m_buffer;
    static uint32_t calculate_crc32(std::span<const std::byte> data);
    bool try_parse_header(uint32_t& out_length, uint32_t& out_crc);
};
```

**Verification Tasks:**

1.  **Zero-Error Parsing across Arbitrary Chunks:** Feed a complete
    $1024$-byte framed message into `FrameDecoder` across randomized
    slice lengths ($1$ byte, $17$ bytes, $256$ bytes). Verify that
    `extract_next_frame()` produces the identical uncorrupted message.
2.  **Bit-Flip Corruption Detection:** Flip a single bit within the
    payload buffer. Verify that `extract_next_frame()` flags a checksum
    mismatch error, discards the corrupted frame, and protects the
    application layer.
3.  **Multi-Frame Coalescing:** Concatenate three complete valid frames
    into a single continuous byte buffer and feed it into the decoder.
    Verify that three sequential calls to `extract_next_frame()` extract
    each message cleanly in order.

#### Exercise 17.4: Heartbeat and Dead-Peer Detection (Keep-Alive State Machine)

**Context and Theory:**

In a TCP connection, if an intermediate network router drops its state
or the remote machine suffers a sudden power loss, no TCP `FIN` or `RST`
packet is ever generated. From the perspective of the local kernel, the
socket remains valid and in state `ESTABLISHED` indefinitely.

To detect these **Half-Open Connections**, applications implement
\*\*Application-Level Heartbeats (see Figure 17–11):

1.  Every $T_{\text{interval}}$ seconds of communication silence, the
    host transmits a light-weight `PING` frame.
2.  If the remote peer does not respond with a `PONG` within
    $T_{\text{timeout}}$ seconds across $N$ consecutive attempts, the
    connection is pronounced dead, and the socket is closed.

<figure id="fig:keep-alive">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0OTkuMiIgaGVp
Z2h0PSIxMjguOCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0OTkuMiAxMjguOCI+CiAg
PHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2Jv
YiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRo
OiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1s
aW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQg
ewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9
Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3Bh
cmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3Zn
Ym9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZp
bGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFy
a2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25k
IHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRf
bWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5z
dGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2Np
cmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0
YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9j
aXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19v
cGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIg
dmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFy
a2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29u
IHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxt
YXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdv
bj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4
IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmll
bnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIy
IiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0i
b3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldp
ZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAg
ICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xl
PgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9k
ZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI0OTkuMiIg
aGVpZ2h0PSIxMjguOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijku
NzUiIHk9IjExLjcwMDAwMSIgPkhvc3Q8L3RleHQ+CiAgPGxpbmUgeDE9IjI3LjMwMDAwMSIgeTE9
IjE1LjYiIHgyPSIyNy4zMDAwMDEiIHkyPSIxMDkuMjAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGlu
ZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0OC43NSIgeT0iMTEuNzAwMDAxIiA+
QTwvdGV4dD4KICA8bGluZSB4MT0iNDYuODAwMDAzIiB5MT0iMjMuNDAwMDAyIiB4Mj0iNzAuMjAw
MDA1IiB5Mj0iMjMuNDAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSI3OS45NTAwMDUiIHk9IjI3LjMwMDAwMSIgPlBJTkc8L3RleHQ+CiAg
PHBvbHlnb24gcG9pbnRzPSI1NC42MDAwMDIsMzUuMTAwMDAyIDQ2LjgwMDAwMywzOSA1NC42MDAw
MDIsNDIuOSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSI1NC42MDAwMDIi
IHkxPSIzOSIgeDI9IjcwLjIwMDAwNSIgeTI9IjM5IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3OS45NTAwMDUiIHk9IjQyLjkiID5QT05HPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ4Ljc1IiB5PSI1OC41IiA+Li4u
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijc5Ljk1MDAwNSIgeT0iNTgu
NSIgPltOZXR3b3JrPC90ZXh0PgogIDxsaW5lIHgxPSI0Ni44MDAwMDMiIHkxPSI3MC4yMDAwMDUi
IHgyPSI3MC4yMDAwMDUiIHkyPSI3MC4yMDAwMDUiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijc5Ljk1MDAwNSIgeT0iNzQuMSIgPlBJTkc8L3Rl
eHQ+CiAgPGxpbmUgeDE9IjQ2LjgwMDAwMyIgeTE9Ijg1LjgiIHgyPSI3MC4yMDAwMDUiIHkyPSI4
NS44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI3OS45NTAwMDUiIHk9Ijg5LjcwMDAwNSIgPlBJTkc8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNDguNzUiIHk9IjEwNS4zIiA+W1RpbWVvdXQ8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDQ2LjU1MDAyIiB5PSIxMS43MDAwMDEiID5Ib3N0PC90
ZXh0PgogIDxsaW5lIHgxPSI0NjQuMSIgeTE9IjE1LjYiIHgyPSI0NjQuMSIgeTI9IjU0LjYwMDAw
MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NDg1LjU1MDAyIiB5PSIxMS43MDAwMDEiID5CPC90ZXh0PgogIDxsaW5lIHgxPSIxOTUiIHkxPSIy
My40MDAwMDIiIHgyPSI0MTMuNDAwMDIiIHkyPSIyMy40MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9s
aW5lPgogIDxwb2x5Z29uIHBvaW50cz0iNDEzLjQwMDAyLDE5LjUgNDIxLjIsMjMuNDAwMDAyIDQx
My40MDAwMiwyNy4zMDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0i
MjE4LjQwMDAxIiB5MT0iMzkiIHgyPSI0MjEuMiIgeTI9IjM5IiBjbGFzcz0ic29saWQiPjwvbGlu
ZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxOTYuOTUwMDEiIHk9IjU4LjUiID5D
dXRdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIzNS45NTAwMSIgeT0i
NTguNSIgPi4uLjwvdGV4dD4KICA8bGluZSB4MT0iMTk1IiB5MT0iNzAuMjAwMDA1IiB4Mj0iNDEz
LjQwMDAyIiB5Mj0iNzAuMjAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBw
b2ludHM9IjQxMy40MDAwMiw2Ni4zIDQyMS4yLDcwLjIwMDAwNSA0MTMuNDAwMDIsNzQuMSIgY2xh
c3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIyMDIuOCIgeTE9Ijg1LjgiIHgyPSI0
MTMuNDAwMDIiIHkyPSI4NS44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjQxMy40MDAwMiw4MS45IDQyMS4yLDg1LjggNDEzLjQwMDAyLDg5LjcwMDAwNSIgY2xhc3M9
ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE5Ni45
NTAwMSIgeT0iMTA1LjMiID5FeGNlZWRlZDwvdGV4dD4KICA8bGluZSB4MT0iMjY1LjIiIHkxPSIx
MDEuNCIgeDI9IjI3MyIgeTI9IjEwMS40IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdv
biBwb2ludHM9IjI3Myw5Ny41IDI4MC44MDAwMiwxMDEuNCAyNzMsMTA1LjMiIGNsYXNzPSJmaWxs
ZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOTAuNTUwMDIi
IHk9IjEwNS4zIiA+QWJvcnQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MzM3LjM1IiB5PSIxMDUuMyIgPkNvbm5lY3Rpb25dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE1MC4xNTAwMSIgeT0iNTguNSIgPkNhYmxlPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjExOC45NTAwMDUiIHk9IjEwNS4zIiA+VGhyZXNob2xkPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExOC45NTAwMDUiIHk9IjI3LjMw
MDAwMSIgPih0ID0gMHMpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEx
OC45NTAwMDUiIHk9IjQyLjkiID4odCA9IDAuMDVzKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIxMTguOTUwMDA1IiB5PSI3NC4xIiA+KHQgPSA1cyk8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDMwLjk1IiB5PSI3NC4xIiA+KGxvc3QpPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExOC45NTAwMDUiIHk9Ijg5LjcwMDAw
NSIgPih0ID0gMTBzKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MzAu
OTUiIHk9Ijg5LjcwMDAwNSIgPihsb3N0KTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 17–11: Application-Level Heartbeats</figcaption>
</figure>

**Specification:**

Construct a thread-safe `HeartbeatMonitor` utilizing C++20
`std::jthread` and condition variables. It must track the timestamp of
the last received valid byte and trigger a forced abort of the
underlying `Socket` descriptor if the timeout threshold is breached.

``` cpp
class HeartbeatMonitor {
public:
    HeartbeatMonitor(Socket& socket, std::chrono::milliseconds interval, 
                     std::chrono::milliseconds timeout);
    void notify_byte_received();
    void start();
    void stop();
private:
    Socket& m_socket;
    std::chrono::milliseconds m_interval, m_timeout;
    std::atomic<int64_t> m_last_seen_epoch_ms{0};
    std::jthread m_monitor_thread;
    void monitor_loop(std::stop_token st);
};
```

**Verification Tasks:**

1.  **Healthy Connection Invariant:** Run a simulation where
    bidirectional traffic flows every $200\text{ ms}$. Set timeout
    threshold to $1,000\text{ ms}$. Verify that the monitor allows the
    connection to persist for $30$ seconds without raising false
    disconnect alerts.
2.  **Dead-Peer Detection:** Halt incoming traffic to simulate peer
    hardware failure. Verify that within $1000 \pm 50\text{ ms}$, the
    monitor terminates the loop, shuts down the socket handle, and
    invokes the disconnect handler.
3.  **Graceful Cancellation:** Stop the monitor cleanly using `stop()`.
    Verify that the internal `std::jthread` exits without dangling
    threads or deadlocks.

#### Exercise 17.5: Token Bucket Network Rate Limiter

**Context and Theory:**

To protect networked database endpoints against Denial of Service (DoS)
attacks and traffic spikes, servers enforce rate limits using the
**Token Bucket Algorithm**.

A token bucket (Figure 17–12) is defined by:

- Capacity ($B$): The maximum number of tokens the bucket can hold (the
  burst limit).
- Refill Rate ($R$): The rate at which tokens are added to the bucket
  (in tokens per second).

When a query frame arrives, the server checks if tokens are available.
If at least $1$ token exists, $1$ token is decremented and the query
proceeds. If the bucket is empty, the request is immediately dropped or
rejected with a rate-limit error frame (`429 Too Many Requests`).

Mathematically, over any time interval $T$, the maximum number of
permitted requests is:

$$M(T) = B + R \cdot T$$

<figure id="fig:token-bucket">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NjgiIGhlaWdo
dD0iMTI4LjgiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNDY4IDEyOC44Ij4KICA8c3R5
bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xlLCAuc3ZnYm9iIHJl
Y3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJva2Utd2lkdGg6IDEu
NTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAgc3Ryb2tlLWxpbmVj
YXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdib2IgdGV4dCB7CiAg
d2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWlseTogQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDEzcHg7Cn0KCi5z
dmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRyYW5zcGFyZW50
Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0KCi5zdmdib2Ig
LmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2ZpbGwgewogIGZp
bGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsKICBtYXJrZXIt
ZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93IHsKICBtYXJr
ZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2RpYW1vbmQgewog
IG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29wZW5fY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6
IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19vcGVuX2NpcmNs
ZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYmlnX29wZW5f
Y2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFycm93IiB2aWV3
Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9p
bnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9wb2x5Z29uPgog
ICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNs
YXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJvcGVu
X2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAg
ICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+
CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjQ2OCIgaGVpZ2h0
PSIxMjguOCI+PC9yZWN0PgogIDxsaW5lIHgxPSI4OS43MDAwMDUiIHkxPSIxNS42IiB4Mj0iODku
NzAwMDA1IiB5Mj0iMzUuMTAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBw
b2ludHM9Ijg1LjgsMzUuMTAwMDAyIDkzLjYwMDAwNiwzNS4xMDAwMDIgODkuNzAwMDA1LDQ2Ljgw
MDAwMyIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjE3LjU1MDAwMSIgeT0iNTguNSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMjQzLjc1IiB5PSI1OC41IiA+XTwvdGV4dD4KICA8bGluZSB4MT0iMjU3LjQi
IHkxPSI1NC42MDAwMDIiIHgyPSIyOTYuNCIgeTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyOTYuNCw1MC43IDMwNC4yLDU0LjYwMDAwMiAyOTYu
NCw1OC41IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzEzLjk1IiB5PSI1OC41IiA+RHJvcDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIzNTIuOTUiIHk9IjU4LjUiID5leGNlc3M8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDA3LjU1MDAyIiB5PSI1OC41IiA+aWY8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDMwLjk1IiB5PSI1OC41IiA+ZnVsbDwvdGV4dD4K
ICA8bGluZSB4MT0iODkuNzAwMDA1IiB5MT0iNjIuNCIgeDI9Ijg5LjcwMDAwNSIgeTI9Ijc4IiBj
bGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNS4z
NSIgeT0iODkuNzAwMDA1IiA+UmVxdWVzdDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI4Ny43NSIgeT0iODkuNzAwMDA1IiA+YXJyaXZlcz88L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjUuMzUiIHk9IjEwNS4zIiA+QnVja2V0PC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijc5Ljk1MDAwNSIgeT0iMTA1LjMiID5lbXB0eT88
L3RleHQ+CiAgPGxpbmUgeDE9IjE1NiIgeTE9Ijg1LjgiIHgyPSIxNzEuNiIgeTI9Ijg1LjgiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTcxLjYsODEuOSAxNzkuNDAw
MDEsODUuOCAxNzEuNiw4OS43MDAwMDUiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGlu
ZSB4MT0iMTU2IiB5MT0iMTAxLjQiIHgyPSIxNzEuNiIgeTI9IjEwMS40IiBjbGFzcz0ic29saWQi
PjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjE3MS42LDk3LjUgMTc5LjQwMDAxLDEwMS40IDE3
MS42LDEwNS4zIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMTg5LjE1MDAxIiB5PSI4OS43MDAwMDUiID5UYWtlPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIyOC4xNTAwMSIgeT0iODkuNzAwMDA1IiA+MTwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxODkuMTUwMDEiIHk9IjEwNS4zIiA+
UmVqZWN0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0My43NSIgeT0i
ODkuNzAwMDA1IiA+dG9rZW48L3RleHQ+CiAgPGxpbmUgeDE9IjI4OC42IiB5MT0iODUuOCIgeDI9
IjMwNC4yIiB5Mj0iODUuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRz
PSIzMDQuMiw4MS45IDMxMiw4NS44IDMwNC4yLDg5LjcwMDAwNSIgY2xhc3M9ImZpbGxlZCI+PC9w
b2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMyMS43NSIgeT0iODkuNzAw
MDA1IiA+UHJvY2VzczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNDMu
NzUiIHk9IjEwNS4zIiA+d2l0aDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIyODIuNzUiIHk9IjEwNS4zIiA+RXJyb3I8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzg0LjE1MDAyIiB5PSI4OS43MDAwMDUiID5SZXF1ZXN0PC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjExLjcwMDAwMSIgPlRva2VuIFNvdXJj
ZSAoK1IgdG9rZW5zL3NlYyk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MzMuMTUiIHk9IjU4LjUiID5Ub2tlbiBCdWNrZXQgKENhcGFjaXR5IEIpPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 17–12: Token Bucket</figcaption>
</figure>

**Specification:**

Implement a thread-safe `TokenBucketLimiter` class. Integrate it into
the `TcpServer` dispatch loop to limit individual client IP addresses to
an agreed quota.

``` cpp
class TokenBucketLimiter {
public:
    TokenBucketLimiter(double refill_rate, double capacity);
    bool try_consume(double tokens = 1.0);
private:
    double m_refill_rate;
    double m_capacity;
    double m_current_tokens;
    std::chrono::steady_clock::time_point m_last_refill;
    std::mutex m_mutex;
    void refill();
};
```

**Verification Tasks:**

1.  **Burst Capacity Conformance:** Initialise with capacity $B = 10$,
    rate $R = 2\text{ tokens/sec}$. Issue $10$ rapid sequential calls to
    `try_consume()`. Verify that all $10$ succeed immediately.
2.  **Instantaneous Throttling:** Issue an $11\text{th}$ immediate
    request following the burst. Verify that `try_consume()` returns
    `false`.
3.  **Temporal Replenishment:** After exhausting the bucket, sleep for
    $1.0\text{ second}$. Verify that exactly $2$ new tokens become
    available for consumption.

#### Exercise 17.6: Thread-Safe Client Connection Pool (RAII Lease Pattern)

**Context and Theory:**

Establishing a TCP connection requires a kernel-level three-way
handshake (`SYN` $\to$ `SYN-ACK` $\to$ `ACK`), consuming network
round-trip time (RTT). For high-frequency query workloads, establishing
and tearing down connections per query introduces latency and burns
ephemeral local ports.

A **Connection Pool** (as shown in Figure 17–13) pre-allocates and
maintains a set of $K$ persistent, active TCP sockets. Threads request a
connection from the pool, use it to transmit requests, and release it
back to the pool upon completion. By wrapping the leased connection in a
move-only **RAII Lease Guard**, returning the connection to the pool
occurs automatically when the handle exits scope.

<figure id="fig:client-pool">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NDYiIGhlaWdo
dD0iNjYuNCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1NDYgNjYuNCI+CiAgPHN0eWxl
Pi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2JvYiByZWN0
LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRoOiAxLjU7
CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1saW5lY2Fw
OiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQgewogIHdo
aXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJpZXIgTmV3
LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9Cgouc3Zn
Ym9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3BhcmVudDsK
fQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3ZnYm9iIC5m
aWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZpbGw6IHRy
YW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFya2VyLWVu
ZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25kIHsKICBt
YXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2NpcmNsZSk7
Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9jaXJjbGUg
ewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19vcGVuX2Np
cmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIgdmlld0Jv
eD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50
cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2Vy
V2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgog
ICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdvbj4KICAg
IDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZY
PSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1
dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFz
cz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0ib3Blbl9j
aXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3
IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNp
cmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAg
PC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4
IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9y
aWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9
IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgog
IDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1NDYiIGhlaWdodD0i
NjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9IjEx
LjcwMDAwMSIgPkNsaWVudDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2
NC4zNSIgeT0iMTEuNzAwMDAxIiA+VGhyZWFkPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjExOC45NTAwMDUiIHk9IjExLjcwMDAwMSIgPjE8L3RleHQ+CiAgPGxpbmUgeDE9
IjEzMi42IiB5MT0iNy44IiB4Mj0iMTYzLjgiIHkyPSI3LjgiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDxwb2x5Z29uIHBvaW50cz0iMTYzLjgsMy45IDE3MS42LDcuOCAxNjMuOCwxMS43MDAwMDEi
IGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIxODEuMzUiIHk9IjExLjcwMDAwMSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMTk2Ljk1MDAxIiB5PSIxMS43MDAwMDEiID5BY3F1aXJlPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1OS4zNSIgeT0iMTEuNzAwMDAxIiA+TGVhc2U8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzA2LjE1IiB5PSIxMS43MDAwMDEi
ID5dPC90ZXh0PgogIDxwb2x5Z29uIHBvaW50cz0iMzY2LjYsMTkuNSAzNzQuNDAwMDIsMjMuNDAw
MDAyIDM2Ni42LDI3LjMwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjM4NC4xNTAwMiIgeT0iMjcuMzAwMDAxIiA+WzwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzOTkuNzUiIHk9IjI3LjMwMDAwMSIgPkxl
YXNlZDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NTQuMzUiIHk9IjI3
LjMwMDAwMSIgPlNvY2tldDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1
MDguOTUiIHk9IjI3LjMwMDAwMSIgPiMyPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjUzMi4zNTAwNCIgeT0iMjcuMzAwMDAxIiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSI5Ljc1IiB5PSI0Mi45IiA+Q2xpZW50PC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjY0LjM1IiB5PSI0Mi45IiA+VGhyZWFkPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExOC45NTAwMDUiIHk9IjQyLjkiID4xPC90ZXh0
PgogIDxwb2x5Z29uIHBvaW50cz0iMTQwLjQwMDAxLDM1LjEwMDAwMiAxMzIuNiwzOSAxNDAuNDAw
MDEsNDIuOSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIxNDAuNDAwMDEi
IHkxPSIzOSIgeDI9IjE3MS42IiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE4MS4zNSIgeT0iNDIuOSIgPls8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzA2LjE1IiB5PSI0Mi45IiA+XTwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxOTYuOTUwMDEiIHk9IjQyLjkiID5BdXRvLVJl
bGVhc2U8L3RleHQ+CiAgPGc+CiAgICA8bGluZSB4MT0iMzE5LjgwMDAyIiB5MT0iNy44IiB4Mj0i
MzQ3LjEiIHkyPSI3LjgiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjM0Ny4x
IiB5MT0iNy44IiB4Mj0iMzQ3LjEiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8
bGluZSB4MT0iMzQ3LjEiIHkxPSIyMy40MDAwMDIiIHgyPSIzNjYuNiIgeTI9IjIzLjQwMDAwMiIg
Y2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMzE5LjgwMDAyIiB5MT0iMzkiIHgy
PSIzNDcuMSIgeTI9IjM5IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 17–13: Client Connection Pool</figcaption>
</figure>

**Specification:**

Implement a `ConnectionPool` that manages a fixed set of persistent
`TcpClient` instances. Construct a `ScopedConnectionLease` RAII wrapper
that releases the socket back into the pool upon destruction.

``` cpp
class ConnectionPool {
public:
    ConnectionPool(std::string host, uint16_t port, size_t pool_size);
    class Lease {
    public:
        Lease(ConnectionPool& pool, std::unique_ptr<TcpClient> client);
        ~Lease();
        TcpClient& operator*() { return *m_client; }
        TcpClient* operator->() { return m_client.get(); }
    private:
        ConnectionPool& m_pool;
        std::unique_ptr<TcpClient> m_client;
    };
    Lease acquire();
private:
    friend class Lease;
    void release(std::unique_ptr<TcpClient> client);
    std::queue<std::unique_ptr<TcpClient>> m_pool;
    std::mutex m_mutex;
    std::condition_variable m_cv;
};
```

**Verification Tasks:**

1.  **Concurrency and Contention:** Launch $20$ worker threads sharing a
    pool of $4$ connections. Have each thread execute $50$ network
    requests. Verify that all $1,000$ operations complete without
    connection failures or race conditions.
2.  **RAII Exception Safety:** Acquire a connection and throw an
    intentional C++ exception before an explicit release. Catch the
    exception and verify that the connection was returned to the pool by
    the `Lease` destructor.
3.  **Exhaustion Blocking:** Acquire all $K$ connections. Attempt an
    acquisition on a $(K+1)\text{th}$ thread with a $100\text{ ms}$
    timeout. Verify that the thread blocks until one of the active
    leases is destructed.

#### Exercise 17.7: Binary Serialisation of Relational Tuples via `std::span`

**Context and Theory:**

Formatting query results as human-readable ASCII text tables (as
demonstrated in Section 17.8) incurs high CPU serialisation overhead and
bloats network payloads. High-performance databases transmit records
using **Binary Row Serialisation**.

A binary tuple frame lays out fields contiguously in memory (as shown in
Figure 17–14):

1.  **Null Bitmask:** A compact bitset indicating which columns hold
    `NULL` values.
2.  **Fixed-Width Column Offsets:** Direct scalar mappings (e.g., 64-bit
    integers and doubles converted to Network Byte Order).
3.  **Variable-Length Heaps:** Prepending string and byte-array
    attributes with their uint32 byte length, followed by their raw
    character bytes.

<figure id="fig:binary-tuple">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI3MjUuNCIgaGVp
Z2h0PSIzNS4yIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDcyNS40IDM1LjIiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNzI1LjQiIGhl
aWdodD0iMzUuMiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9IjExLjcwMDAwMSIgPlsgTnVsbCBNYXNrOiAxQiBdWyBDb2wgMCAoSW50NjQpOiA4QiBdWyBD
b2wgMSAoRmxvYXQ2NCk6IDhCIF1bIFN0ciBMZW46IDRCIF1bIFN0ciBCeXRlczogTiBdPC90ZXh0
Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 17–14: Binary Tuple Layout</figcaption>
</figure>

**Specification:**

Implement binary serialisation and deserialisation routines for
`CoreDB::Row` and `CoreDB::Value` (from Chapter 16) targeting network
byte buffers. Use `std::span` and endian-safe operations
(`std::endian`).

``` cpp
class TupleWireFormat {
public:
    static std::vector<std::byte> serialise_row(const Row& row, const Schema& schema);
    static Row deserialise_row(std::span<const std::byte> buffer, const Schema& schema);
private:
    static void write_uint64(std::vector<std::byte>& dest, uint64_t val);
    static uint64_t read_uint64(std::span<const std::byte>& src);
};
```

**Verification Tasks:**

1.  **Round-Trip Value Fidelity:** Serialise a row containing
    `(42, 3.14159, "Ada Lovelace", NULL)`. Deserialise the resulting
    binary buffer and verify that types, numerical values, and string
    contents match the original tuple exactly.
2.  **Bandwidth Reduction Benchmark:** Compare the size of a
    binary-serialised batch of $10,000$ rows against an equivalent
    JSON/ASCII representation. Verify that binary formatting reduces
    network footprint by at least $60\%$.
3.  **Truncated Buffer Safety:** Slice a binary buffer in half to
    simulate a truncated network packet. Pass the truncated slice to
    `deserialise_row`. Verify that the routine detects buffer underflow
    and throws an exception rather than reading past buffer boundaries.

#### Exercise 17.8: Chunked Streaming Protocol for Massive Query Results

**Context and Theory:**

If a query matches $10,000,000$ rows, buffering the entire result set in
server RAM before sending crashes the process with out-of-memory (OOM)
errors.

To solve this, modern engines use a **Chunked Streaming Protocol** (see
Figure 17–15):

1.  The server executes the query iterator pipeline in batches (e.g.,
    $1,000$ tuples per chunk).
2.  Each batch is serialised into an independent binary frame with a
    header flag: `FLAG_CONTINUATION` ($0\text{x}01$) or `FLAG_FINAL`
    ($0\text{x}02$).
3.  The client receives, processes, and releases chunks sequentially,
    maintaining a constant memory footprint regardless of dataset size.

<figure id="fig:chunked-streaming">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MjQiIGhlaWdo
dD0iODIiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNjI0IDgyIj4KICA8c3R5bGU+LnN2
Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xlLCAuc3ZnYm9iIHJlY3QsIC5z
dmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJva2Utd2lkdGg6IDEuNTsKICBz
dHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAgc3Ryb2tlLWxpbmVjYXA6IHJv
dW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdib2IgdGV4dCB7CiAgd2hpdGUt
c3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWlseTogQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDEzcHg7Cn0KCi5zdmdib2Ig
cmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRyYW5zcGFyZW50Owp9Cgou
c3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0KCi5zdmdib2IgLmZpbGxl
ZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7CiAgZmlsbDogdHJhbnNw
YXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2ZpbGwgewogIGZpbGw6IHRy
YW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsKICBtYXJrZXItZW5kOiB1
cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93IHsKICBtYXJrZXItc3Rh
cnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtl
ci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9kaWFtb25kIHsK
ICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtl
ZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRf
bWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29wZW5fY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAg
bWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYmlnX29wZW5fY2lyY2xl
KTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFycm93IiB2aWV3Qm94PSIt
MiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIw
LDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0i
ZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9wb2x5Z29uPgogICAgPC9t
YXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQi
IHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1z
dGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJm
aWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJvcGVuX2NpcmNs
ZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1h
cmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xl
IGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21h
cmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMyIg
Y2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+CiAgPHJl
Y3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjYyNCIgaGVpZ2h0PSI4MiI+
PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9IjExLjcwMDAw
MSIgPkNsaWVudDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NC4zNSIg
eT0iMTEuNzAwMDAxIiA+UmVxdWVzdDwvdGV4dD4KICA8bGluZSB4MT0iMTI0LjgiIHkxPSI3Ljgi
IHgyPSIxNDAuNDAwMDEiIHkyPSI3LjgiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29u
IHBvaW50cz0iMTQwLjQwMDAxLDMuOSAxNDguMiw3LjggMTQwLjQwMDAxLDExLjcwMDAwMSIgY2xh
c3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE1
Ny45NSIgeT0iMTEuNzAwMDAxIiA+U2VydmVyPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjIxMi41NSIgeT0iMTEuNzAwMDAxIiA+aW5pdGlhdGVzPC90ZXh0PgogIDxwb2x5
Z29uIHBvaW50cz0iMTc5LjQwMDAxLDE5LjUgMTg3LjIwMDAxLDIzLjQwMDAwMiAxNzkuNDAwMDEs
MjcuMzAwMDAxIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMTk2Ljk1MDAxIiB5PSIyNy4zMDAwMDEiID5FbWl0PC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIzNS45NTAwMSIgeT0iMjcuMzAwMDAxIiA+Q2h1bms8
L3RleHQ+CiAgPHBvbHlnb24gcG9pbnRzPSIxNzkuNDAwMDEsMzUuMTAwMDAyIDE4Ny4yMDAwMSwz
OSAxNzkuNDAwMDEsNDIuOSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjE5Ni45NTAwMSIgeT0iNDIuOSIgPkVtaXQ8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjM1Ljk1MDAxIiB5PSI0Mi45IiA+Q2h1bms8L3Rl
eHQ+CiAgPHBvbHlnb24gcG9pbnRzPSIxNzkuNDAwMDEsNTAuNyAxODcuMjAwMDEsNTQuNjAwMDAy
IDE3OS40MDAwMSw1OC41IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMTk2Ljk1MDAxIiB5PSI1OC41IiA+RW1pdDwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMzUuOTUwMDEiIHk9IjU4LjUiID5DaHVuazwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOTAuNTUwMDIiIHk9IjExLjcwMDAw
MSIgPlRhYmxlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMzNy4zNSIg
eT0iMTEuNzAwMDAxIiA+U2NhbjwvdGV4dD4KICA8bGluZSB4MT0iNDI5IiB5MT0iMjMuNDAwMDAy
IiB4Mj0iNDQ0LjYiIHkyPSIyMy40MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5
Z29uIHBvaW50cz0iNDQ0LjYsMTkuNSA0NTIuNDAwMDIsMjMuNDAwMDAyIDQ0NC42LDI3LjMwMDAw
MSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSI0MjkiIHkxPSIzOSIgeDI9
IjQ0NC42IiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0i
NDQ0LjYsMzUuMTAwMDAyIDQ1Mi40MDAwMiwzOSA0NDQuNiw0Mi45IiBjbGFzcz0iZmlsbGVkIj48
L3BvbHlnb24+CiAgPGxpbmUgeDE9IjQyOSIgeTE9IjU0LjYwMDAwMiIgeDI9IjQ0NC42IiB5Mj0i
NTQuNjAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjQ0NC42
LDUwLjcgNDUyLjQwMDAyLDU0LjYwMDAwMiA0NDQuNiw1OC41IiBjbGFzcz0iZmlsbGVkIj48L3Bv
bHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDYyLjE1MDAyIiB5PSIyNy4z
MDAwMDEiID5DbGllbnQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDYy
LjE1MDAyIiB5PSI0Mi45IiA+Q2xpZW50PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjQ2Mi4xNTAwMiIgeT0iNTguNSIgPkNsaWVudDwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSI1MTYuNzUiIHk9IjI3LjMwMDAwMSIgPmNvbnN1bWVzPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUxNi43NSIgeT0iNDIuOSIgPmNvbnN1bWVz
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUxNi43NSIgeT0iNTguNSIg
PmNsb3NlczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1NzEuMzUwMDQi
IHk9IjU4LjUiID5zdHJlYW08L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MjgyLjc1IiB5PSIyNy4zMDAwMDEiID4jMSBbQ29udGludWF0aW9uXTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODIuNzUiIHk9IjQyLjkiID4jMiBbQ29udGludWF0aW9u
XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODIuNzUiIHk9IjU4LjUi
ID4jMyBbRmluYWxdPC90ZXh0PgogIDxnPgogICAgPGxpbmUgeDE9IjE1OS45MDAwMSIgeTE9IjE1
LjYiIHgyPSIxNTkuOTAwMDEiIHkyPSI2Mi40IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxs
aW5lIHgxPSIxNTkuOTAwMDEiIHkxPSIyMy40MDAwMDIiIHgyPSIxNzkuNDAwMDEiIHkyPSIyMy40
MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjE1OS45MDAwMSIgeTE9
IjM5IiB4Mj0iMTc5LjQwMDAxIiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxp
bmUgeDE9IjE1OS45MDAwMSIgeTE9IjU0LjYwMDAwMiIgeDI9IjE3OS40MDAwMSIgeTI9IjU0LjYw
MDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 17–15: Chunked Streaming</figcaption>
</figure>

**Specification:**

Construct a `ChunkedStreamWriter` and `ChunkedStreamReader`. Integrate
the reader and writer with the Chapter 16 `ExecutionNode` iterator
interface so that massive tables stream across the network in bounded
chunks.

``` cpp
class ChunkedStreamWriter {
public:
    ChunkedStreamWriter(Socket& socket, size_t batch_size);
    void write_from_node(ExecutionNode& plan);
private:
    Socket& m_socket;
    size_t m_batch_size;
    void send_chunk(const std::vector<Row>& rows, bool is_final);
};

class ChunkedStreamReader : public ExecutionNode {
public:
    explicit ChunkedStreamReader(Socket& socket, Schema schema);
    void open() override;
    std::optional<Row> next() override;
    const Schema& output_schema() const override { return m_schema; }
private:
    Socket& m_socket;
    Schema m_schema;
    std::vector<Row> m_current_chunk;
    size_t m_chunk_cursor{0};
    bool m_stream_ended{false};
    void fetch_next_chunk();
};
```

**Verification Tasks:**

1.  **Constant Memory Invariant:** Stream a synthetic dataset of
    $1,000,000$ rows across a local socket using batch size $500$.
    Monitor process resident memory (RSS); verify that memory
    consumption does not exceed $20\text{ MB}$ over the entire run.
2.  **Stream Reconstruction:** Compare the original row sequence on the
    server with the output yielded by `ChunkedStreamReader::next()`.
    Verify that all rows match without omissions or reordering.
3.  **Empty Result Stream:** Execute a query returning zero rows. Verify
    that the server sends a single chunk marked `FLAG_FINAL` containing
    zero rows, and that the client's `next()` returns `std::nullopt` on
    its first invocation.

#### Exercise 17.9: Stop-and-Wait ARQ over Unreliable Datagrams

**Context and Theory:**

While TCP handles reliability at the kernel level, embedded systems and
high-speed telemetry applications often require custom reliability
layers over UDP using **Automatic Repeat reQuest (ARQ)**.

The simplest reliable protocol is **Stop-and-Wait ARQ (Alternating Bit
Protocol)** (as shown in Figure 17–16):

1.  The sender transmits a frame tagged with a 1-bit sequence number
    ($s \in \{0, 1\}$) and starts a timer.
2.  The sender stops and waits: it will not transmit the next frame
    until it receives an ACK with matching number $a = s$.
3.  If the timer expires before the ACK arrives, the sender retransmits
    the same frame.
4.  If the receiver receives a duplicate frame with the same sequence
    bit, it acknowledges the frame but discards the duplicate payload.

<figure id="fig:stop-and-wait">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSIzNzQuNDAwMDIi
IGhlaWdodD0iMjIyLjQwMDAxIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDM3NC40MDAw
MiAyMjIuNDAwMDEiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2Jv
YiBjaXJjbGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNr
OwogIHN0cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0
eTogMTsKICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsK
fQoKLnN2Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZv
bnQtZmFtaWx5OiBDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZv
bnQtc2l6ZTogMTNweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hh
cnJheTogODsKfQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAu
YmdfZmlsbGVkIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5z
dmdib2IgLm5vZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfYXJyb3cgewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfYXJyb3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVu
ZF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1l
bmQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJj
bGUgewogIG1hcmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9t
YXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNs
ZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1h
cmtlciBpZD0iYXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFy
a2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2Ui
PgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAg
PC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAy
LDQgMCwyIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiBy
ZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdf
ZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5f
Y2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxj
aXJjbGUgY3g9IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAg
IDwvbWFya2VyPgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00
IiB3aWR0aD0iMzc0LjQwMDAyIiBoZWlnaHQ9IjIyMi40MDAwMSI+PC9yZWN0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjExLjcwMDAwMSIgPlNlbmRlcjwvdGV4dD4K
ICA8bGluZSB4MT0iMjcuMzAwMDAxIiB5MT0iMTUuNiIgeDI9IjI3LjMwMDAwMSIgeTI9IjIwMi44
IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8bGluZSB4MT0iMzkiIHkxPSIyMy40MDAwMDIiIHgy
PSI2Mi40IiB5Mj0iMjMuNDAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIzMDYuMTUiIHk9IjExLjcwMDAwMSIgPlJlY2VpdmVyPC90ZXh0
PgogIDxsaW5lIHgxPSIyMDIuOCIgeTE9IjIzLjQwMDAwMiIgeDI9IjMxMiIgeTI9IjIzLjQwMDAw
MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIzMTIsMTkuNSAzMTku
ODAwMDIsMjMuNDAwMDAyIDMxMiwyNy4zMDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4K
ICA8bGluZSB4MT0iMzMxLjUiIHkxPSIxNS42IiB4Mj0iMzMxLjUiIHkyPSIyMDIuOCIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI0Ni44MDAwMDMsNTAuNyAzOSw1NC42
MDAwMDIgNDYuODAwMDAzLDU4LjUiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4
MT0iNDYuODAwMDAzIiB5MT0iNTQuNjAwMDAyIiB4Mj0iNjIuNCIgeTI9IjU0LjYwMDAwMiIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9IjE4Ny4yMDAwMSIgeTE9IjU0LjYwMDAwMiIg
eDI9IjMxOS44MDAwMiIgeTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPGxp
bmUgeDE9IjM5IiB5MT0iODUuOCIgeDI9IjYyLjQiIHkyPSI4NS44IiBjbGFzcz0ic29saWQiPjwv
bGluZT4KICA8bGluZSB4MT0iMjAyLjgiIHkxPSI4NS44IiB4Mj0iMjE4LjQwMDAxIiB5Mj0iODUu
OCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyMTguNDAwMDEsODEu
OSAyMjYuMjAwMDEsODUuOCAyMTguNDAwMDEsODkuNzAwMDA1IiBjbGFzcz0iZmlsbGVkIj48L3Bv
bHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjM1Ljk1MDAxIiB5PSI4OS43
MDAwMDUiID5bRHJvcHBlZF08L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NDAuOTUiIHk9IjEyMC45IiA+W1RpbWVyPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9Ijk1LjU1IiB5PSIxMjAuOSIgPkV4cGlyZXNdPC90ZXh0PgogIDxsaW5lIHgxPSIzOSIg
eTE9IjE0OC4yIiB4Mj0iNjIuNCIgeTI9IjE0OC4yIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8
bGluZSB4MT0iMjAyLjgiIHkxPSIxNDguMiIgeDI9IjMxMiIgeTI9IjE0OC4yIiBjbGFzcz0ic29s
aWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjMxMiwxNDQuMyAzMTkuODAwMDIsMTQ4LjIg
MzEyLDE1Mi4xIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHBvbHlnb24gcG9pbnRzPSI0
Ni44MDAwMDMsMTc1LjUgMzksMTc5LjQwMDAxIDQ2LjgwMDAwMywxODMuMyIgY2xhc3M9ImZpbGxl
ZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSI0Ni44MDAwMDMiIHkxPSIxNzkuNDAwMDEiIHgyPSI2
Mi40IiB5Mj0iMTc5LjQwMDAxIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8bGluZSB4MT0iMTg3
LjIwMDAxIiB5MT0iMTc5LjQwMDAxIiB4Mj0iMzE5LjgwMDAyIiB5Mj0iMTc5LjQwMDAxIiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3Mi4xNSIg
eT0iMjcuMzAwMDAxIiA+RnJhbWUgKHNlcSA9IDApPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjcyLjE1IiB5PSI1OC41IiA+QUNLIChzZXEgPSAwKTwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3Mi4xNSIgeT0iODkuNzAwMDA1IiA+RnJhbWUgKHNl
cSA9IDEpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcyLjE1IiB5PSIx
NTIuMSIgPkZyYW1lIChzZXEgPSAxKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSI3Mi4xNSIgeT0iMTgzLjMiID5BQ0sgKHNlcSA9IDEpPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 17–16: Stop-and-Wait ARQ (Alternating Bit
Protocol)</figcaption>
</figure>

**Specification:**

Implement an alternating-bit `ReliableUdpChannel` class that
encapsulates a raw UDP socket, guaranteeing reliable transmission and
duplicate rejection over an unreliable, packet-dropping link.

``` cpp
class ReliableUdpChannel {
public:
    ReliableUdpChannel(Socket sock, sockaddr_in peer, std::chrono::milliseconds timeout);
    void send_reliable(std::span<const std::byte> payload);
    std::vector<std::byte> receive_reliable();
private:
    Socket m_sock;
    sockaddr_in m_peer;
    std::chrono::milliseconds m_timeout;
    bool m_send_seq_bit{false};
    bool m_expected_recv_bit{false};
    void wait_for_ack(bool expected_bit);
};
```

**Verification Tasks:**

1.  **Packet Loss Recovery:** Introduce an artificial loss proxy that
    drops every second datagram. Transmit $50$ messages. Verify that all
    $50$ messages arrive safely at the receiver in exact sequence.
2.  **Duplicate Discard Invariant:** Intentionally delay ACKs to trigger
    spurious retransmissions. Verify that the receiving application
    layer receives only one copy of each payload.
3.  **Timeout Adaptability:** Verify that altering the timeout parameter
    from $50\text{ ms}$ to $500\text{ ms}$ scales the retransmission
    delay accordingly.

#### Exercise 17.10: Distributed Two-Phase Commit (2PC) Coordinator

**Context and Theory:**

When a database is sharded across multiple physical network nodes,
maintaining the **Atomicity ($A$)** of transactions requires distributed
consensus. If Node 1 commits an insert but Node 2 fails due to a
constraint violation, the distributed transaction must roll back
globally.

The standard consensus protocol is the **Two-Phase Commit (2PC)**:

- **Phase 1: Prepare (Voting):**
  1.  The Coordinator transmits a `PREPARE` frame to all Participant
      nodes.
  2.  Each participant checks constraints, writes undo/redo logs, and
      replies with `VOTE_COMMIT` or `VOTE_ABORT`.
- **Phase 2: Commit / Abort (Decision):**
  1.  If **every** participant votes `VOTE_COMMIT`, the coordinator
      sends `GLOBAL_COMMIT`.
  2.  If **any** participant votes `VOTE_ABORT` (or times out), the
      coordinator sends `GLOBAL_ABORT`.

<figure id="two-phase-commit">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NTMuOCIgaGVp
Z2h0PSIyMzgiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNTUzLjggMjM4Ij4KICA8c3R5
bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xlLCAuc3ZnYm9iIHJl
Y3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJva2Utd2lkdGg6IDEu
NTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAgc3Ryb2tlLWxpbmVj
YXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdib2IgdGV4dCB7CiAg
d2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWlseTogQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDEzcHg7Cn0KCi5z
dmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRyYW5zcGFyZW50
Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0KCi5zdmdib2Ig
LmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2ZpbGwgewogIGZp
bGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsKICBtYXJrZXIt
ZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93IHsKICBtYXJr
ZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2RpYW1vbmQgewog
IG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29wZW5fY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6
IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19vcGVuX2NpcmNs
ZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYmlnX29wZW5f
Y2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFycm93IiB2aWV3
Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9p
bnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9wb2x5Z29uPgog
ICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNs
YXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJvcGVu
X2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAg
ICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+
CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjU1My44IiBoZWln
aHQ9IjIzOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9
IjExLjcwMDAwMSIgPkNvb3JkaW5hdG9yPC90ZXh0PgogIDxsaW5lIHgxPSI1MC43IiB5MT0iMTUu
NiIgeDI9IjUwLjciIHkyPSIyMTguNDAwMDEiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxsaW5l
IHgxPSI2Mi40IiB5MT0iMjMuNDAwMDAyIiB4Mj0iODUuOCIgeTI9IjIzLjQwMDAwMiIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTUuNTUiIHk9
IjI3LjMwMDAwMSIgPlBSRVBBUkU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMjQzLjc1IiB5PSIxMS43MDAwMDEiID5QYXJ0aWNpcGFudDwvdGV4dD4KICA8bGluZSB4MT0i
MTU2IiB5MT0iMjMuNDAwMDAyIiB4Mj0iMjczIiB5Mj0iMjMuNDAwMDAyIiBjbGFzcz0ic29saWQi
PjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjI3MywxOS41IDI4MC44MDAwMiwyMy40MDAwMDIg
MjczLDI3LjMwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIyOTIu
NSIgeTE9IjE1LjYiIHgyPSIyOTIuNSIgeTI9IjQ2LjgwMDAwMyIgY2xhc3M9InNvbGlkIj48L2xp
bmU+CiAgPGxpbmUgeDE9IjE1NiIgeTE9IjU0LjYwMDAwMiIgeDI9IjQ3NS44MDAwMiIgeTI9IjU0
LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI0NzUuODAw
MDIsNTAuNyA0ODMuNiw1NC42MDAwMDIgNDc1LjgwMDAyLDU4LjUiIGNsYXNzPSJmaWxsZWQiPjwv
cG9seWdvbj4KICA8bGluZSB4MT0iMjkyLjUiIHkxPSI2Mi40IiB4Mj0iMjkyLjUiIHkyPSIxMDku
MjAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8bGluZSB4MT0iMTc5LjQwMDAxIiB5MT0i
MTE3IiB4Mj0iNDgzLjYiIHkyPSIxMTciIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxsaW5lIHgx
PSIyOTIuNSIgeTE9IjEyNC44IiB4Mj0iMjkyLjUiIHkyPSIxNDAuNDAwMDEiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogIDxsaW5lIHgxPSIyODAuODAwMDIiIHkxPSIxNDguMiIgeDI9IjI5Ni40IiB5
Mj0iMTQ4LjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMjk2LjQs
MTQ0LjMgMzA0LjIsMTQ4LjIgMjk2LjQsMTUyLjEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4K
ICA8bGluZSB4MT0iMjkyLjUiIHkxPSIxNTYiIHgyPSIyOTIuNSIgeTI9IjE4Ny4yMDAwMSIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzM3LjM1
IiB5PSIxMS43MDAwMDEiID4xPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjQ0Ni41NTAwMiIgeT0iMTEuNzAwMDAxIiA+UGFydGljaXBhbnQ8L3RleHQ+CiAgPGxpbmUgeDE9
IjQ5NS4zMDAwMiIgeTE9IjE1LjYiIHgyPSI0OTUuMzAwMDIiIHkyPSIyMTguNDAwMDEiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU0MC4xNSIg
eT0iMTEuNzAwMDAxIiA+MjwvdGV4dD4KICA8bGluZSB4MT0iNjIuNCIgeTE9IjU0LjYwMDAwMiIg
eDI9Ijg1LjgiIHkyPSI1NC42MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9Ijk1LjU1IiB5PSI1OC41IiA+UFJFUEFSRTwvdGV4dD4KICA8
cG9seWdvbiBwb2ludHM9IjcwLjIwMDAwNSw4MS45IDYyLjQsODUuOCA3MC4yMDAwMDUsODkuNzAw
MDA1IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjcwLjIwMDAwNSIgeTE9
Ijg1LjgiIHgyPSI4NS44IiB5Mj0iODUuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTUuNTUiIHk9Ijg5LjcwMDAwNSIgPlZPVEU8L3RleHQ+
CiAgPGxpbmUgeDE9IjEyNC44IiB5MT0iOTMuNjAwMDA2IiB4Mj0iMTMyLjYiIHkyPSI5My42MDAw
MDYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjEzNC41NSIgeT0iODkuNzAwMDA1IiA+Q09NTUlUPC90ZXh0PgogIDxsaW5lIHgxPSIxODcuMjAw
MDEiIHkxPSI4NS44IiB4Mj0iMjgwLjgwMDAyIiB5Mj0iODUuOCIgY2xhc3M9InNvbGlkIj48L2xp
bmU+CiAgPHBvbHlnb24gcG9pbnRzPSI3MC4yMDAwMDUsMTEzLjEwMDAwNiA2Mi40LDExNyA3MC4y
MDAwMDUsMTIwLjkiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iNzAuMjAw
MDA1IiB5MT0iMTE3IiB4Mj0iODUuOCIgeTI9IjExNyIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTUuNTUiIHk9IjEyMC45IiA+Vk9URTwvdGV4
dD4KICA8bGluZSB4MT0iMTI0LjgiIHkxPSIxMjQuOCIgeDI9IjEzMi42IiB5Mj0iMTI0LjgiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEzNC41
NSIgeT0iMTIwLjkiID5BQk9SVDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI2NC4zNSIgeT0iMTUyLjEiID5bVW5hbmltb3VzPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE1MC4xNTAwMSIgeT0iMTUyLjEiID5Db25zZW5zdXM8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjI4LjE1MDAxIiB5PSIxNTIuMSIgPkZhaWxlZDwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMTMuOTUiIHk9IjE1Mi4xIiA+
QWJvcnRdPC90ZXh0PgogIDxsaW5lIHgxPSI2Mi40IiB5MT0iMTc5LjQwMDAxIiB4Mj0iODUuOCIg
eTI9IjE3OS40MDAwMSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iOTUuNTUiIHk9IjE4My4zIiA+R0xPQkFMPC90ZXh0PgogIDxsaW5lIHgxPSIx
NDAuNDAwMDEiIHkxPSIxODcuMjAwMDEiIHgyPSIxNDguMiIgeTI9IjE4Ny4yMDAwMSIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTUwLjE1MDAx
IiB5PSIxODMuMyIgPkFCT1JUPC90ZXh0PgogIDxsaW5lIHgxPSIxOTUiIHkxPSIxNzkuNDAwMDEi
IHgyPSIyNzMiIHkyPSIxNzkuNDAwMDEiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29u
IHBvaW50cz0iMjczLDE3NS41IDI4MC44MDAwMiwxNzkuNDAwMDEgMjczLDE4My4zIiBjbGFzcz0i
ZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjYyLjQiIHkxPSIyMTAuNiIgeDI9Ijg1Ljgi
IHkyPSIyMTAuNiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iOTUuNTUiIHk9IjIxNC41IiA+R0xPQkFMPC90ZXh0PgogIDxsaW5lIHgxPSIxNDAu
NDAwMDEiIHkxPSIyMTguNDAwMDEiIHgyPSIxNDguMiIgeTI9IjIxOC40MDAwMSIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTUwLjE1MDAxIiB5
PSIyMTQuNSIgPkFCT1JUPC90ZXh0PgogIDxsaW5lIHgxPSIxOTUiIHkxPSIyMTAuNiIgeDI9IjQ3
NS44MDAwMiIgeTI9IjIxMC42IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjQ3NS44MDAwMiwyMDYuNzAwMDEgNDgzLjYsMjEwLjYgNDc1LjgwMDAyLDIxNC41IiBjbGFz
cz0iZmlsbGVkIj48L3BvbHlnb24+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Two-Phase Commit</figcaption>
</figure>

**Specification:**

Implement a `TwoPhaseCoordinator` class that orchestrates transactions
across multiple independent networked database servers over TCP frames.

``` cpp
enum class Vote { Commit, Abort };

class TwoPhaseCoordinator {
public:
    void register_participant(Socket sock);
    bool execute_transaction(const std::string& sql_mutation);
private:
    std::vector<Socket> m_participants;
    bool phase_one_prepare();
    void phase_two_decide(bool commit);
};
```

**Verification Tasks:**

1.  **Unanimous Commit:** Register three participant nodes. Execute an
    insert query that satisfies constraints on all nodes. Verify that
    all three nodes execute the mutation and commit successfully.
2.  **Single-Node Failure Abort:** Configure Participant 3 to fail a
    primary key constraint and return `VOTE_ABORT`. Verify that the
    coordinator broadcasts `GLOBAL_ABORT` and that Participants 1 and 2
    roll back their staged mutations.
3.  **Network Timeout Abort:** Disconnect Participant 2 during Phase 1.
    Verify that the coordinator aborts the global transaction after a
    timeout rather than hanging indefinitely.

#### Exercise 17.11: TLS/SSL Handshake and Secure Framing Architecture

**Context and Theory:**

Plaintext TCP sockets are vulnerable to eavesdropping and
Man-In-The-Middle (MITM) tampering. **Transport Layer Security (TLS
1.3)** secures sockets by establishing an encrypted tunnel directly
above the Transport layer.

A TLS session (see Figure 17–17) enforces:

1.  **Authentication:** Using X.509 asymmetric digital certificates to
    verify server identity.
2.  **Key Exchange:** Using Ephemeral Diffie-Hellman (ECDHE) to compute
    a shared symmetric secret key without transmitting the key over the
    wire (achieving **Forward Secrecy**).
3.  **Encryption and Integrity:** Encapsulating application records with
    symmetric ciphers (e.g., AES-256-GCM or ChaCha20-Poly1305) which
    combine encryption with authenticated checksum tags (AEAD).

<figure id="fig:tsl-ssl-handshake">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0ODMuNiIgaGVp
Z2h0PSIxOTEuMjAwMDEiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNDgzLjYgMTkxLjIw
MDAxIj4KICA8c3R5bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xl
LCAuc3ZnYm9iIHJlY3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJv
a2Utd2lkdGg6IDEuNTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAg
c3Ryb2tlLWxpbmVjYXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdi
b2IgdGV4dCB7CiAgd2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWls
eTogQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6
IDEzcHg7Cn0KCi5zdmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6
IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7
Cn0KCi5zdmdib2IgLmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxl
ZCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5u
b2ZpbGwgewogIGZpbGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93
IHsKICBtYXJrZXItZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fy
cm93IHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9kaWFtb25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUp
Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwo
I29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBt
YXJrZXItc3RhcnQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Jp
Z19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVy
bCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9
ImFycm93IiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRo
PSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAg
PHBvbHlnb24gcG9pbnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2Vy
PgogICAgPG1hcmtlciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiBy
ZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+
PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjIiIGNsYXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJvcGVuX2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+
PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtl
cj4KICA8L2RlZnM+CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9
IjQ4My42IiBoZWlnaHQ9IjE5MS4yMDAwMSI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEuOTUiIHk9IjExLjcwMDAwMSIgPkNsaWVudDwvdGV4dD4KICA8bGluZSB4MT0i
MjcuMzAwMDAxIiB5MT0iMTUuNiIgeDI9IjI3LjMwMDAwMSIgeTI9IjE3MS42IiBjbGFzcz0ic29s
aWQiPjwvbGluZT4KICA8bGluZSB4MT0iMzkiIHkxPSIyMy40MDAwMDIiIHgyPSI2Mi40IiB5Mj0i
MjMuNDAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI0MzAuOTUiIHk9IjExLjcwMDAwMSIgPlNlcnZlcjwvdGV4dD4KICA8bGluZSB4MT0i
MzI3LjYiIHkxPSIyMy40MDAwMDIiIHgyPSI0MjkiIHkyPSIyMy40MDAwMDIiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iNDI5LDE5LjUgNDM2LjgwMDAyLDIzLjQwMDAw
MiA0MjksMjcuMzAwMDAxIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjQ0
OC41IiB5MT0iMTUuNiIgeDI9IjQ0OC41IiB5Mj0iMTcxLjYiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcyLjE1IiB5PSIyNy4zMDAwMDEiID5D
bGllbnRIZWxsbzwvdGV4dD4KICA8cG9seWdvbiBwb2ludHM9IjQ2LjgwMDAwMyw1MC43IDM5LDU0
LjYwMDAwMiA0Ni44MDAwMDMsNTguNSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5l
IHgxPSI0Ni44MDAwMDMiIHkxPSI1NC42MDAwMDIiIHgyPSI3MC4yMDAwMDUiIHkyPSI1NC42MDAw
MDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
Ijc5Ljk1MDAwNSIgeT0iNTguNSIgPlNlcnZlckhlbGxvPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjE3My41NSIgeT0iNTguNSIgPis8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMTg5LjE1MDAxIiB5PSI1OC41IiA+Q2VydGlmaWNhdGU8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjgyLjc1IiB5PSI1OC41IiA+KzwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOTguMzUiIHk9IjU4LjUiID5TZXJ2
ZXJLZXlTaGFyZTwvdGV4dD4KICA8bGluZSB4MT0iNDEzLjQwMDAyIiB5MT0iNTQuNjAwMDAyIiB4
Mj0iNDM2LjgwMDAyIiB5Mj0iNTQuNjAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MC45NSIgeT0iODkuNzAwMDA1IiA+W0NsaWVudDwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMDMuMzUwMDA2IiB5PSI4OS43
MDAwMDUiID52ZXJpZmllczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIx
NzMuNTUiIHk9Ijg5LjcwMDAwNSIgPkNlcnQsPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjIyMC4zNSIgeT0iODkuNzAwMDA1IiA+Q29tcHV0ZXM8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjkwLjU1MDAyIiB5PSI4OS43MDAwMDUiID5TaGFyZWQ8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzQ1LjE1IiB5PSI4OS43MDAw
MDUiID5TZWNyZXRdPC90ZXh0PgogIDxsaW5lIHgxPSIzOSIgeTE9IjExNyIgeDI9IjYyLjQiIHky
PSIxMTciIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjcyLjE1IiB5PSIxMjAuOSIgPkNsaWVudEtleVNoYXJlPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjE4OS4xNTAwMSIgeT0iMTIwLjkiID4rPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIwNC43NSIgeT0iMTIwLjkiID5FbmNyeXB0ZWRGaW5p
c2hlZDwvdGV4dD4KICA8bGluZSB4MT0iMzQzLjIiIHkxPSIxMTciIHgyPSI0MjkiIHkyPSIxMTci
IGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iNDI5LDExMy4xMDAwMDYg
NDM2LjgwMDAyLDExNyA0MjksMTIwLjkiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8cG9s
eWdvbiBwb2ludHM9IjQ2LjgwMDAwMywxNDQuMyAzOSwxNDguMiA0Ni44MDAwMDMsMTUyLjEiIGNs
YXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iNDYuODAwMDAzIiB5MT0iMTQ4LjIi
IHgyPSI3MC4yMDAwMDUiIHkyPSIxNDguMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzkuOTUwMDA1IiB5PSIxNTIuMSIgPlNlY3VyZTwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMzQuNTUiIHk9IjE1Mi4xIiA+QUVB
RDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNzMuNTUiIHk9IjE1Mi4x
IiA+RW5jcnlwdGVkPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1MS41
NSIgeT0iMTUyLjEiID5SZWNvcmQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMzA2LjE1IiB5PSIxNTIuMSIgPlN0cmVhbTwvdGV4dD4KICA8bGluZSB4MT0iMzU4LjgwMDAy
IiB5MT0iMTQ4LjIiIHgyPSI0MjkiIHkyPSIxNDguMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAg
PHBvbHlnb24gcG9pbnRzPSI0MjksMTQ0LjMgNDM2LjgwMDAyLDE0OC4yIDQyOSwxNTIuMSIgY2xh
c3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE2
NS43NSIgeT0iMjcuMzAwMDAxIiA+KFN1cHBvcnRlZCBDaXBoZXJzKTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 17–17: TLS/SSL Handshake</figcaption>
</figure>

**Specification:**

Construct a conceptual `SecureSocket` RAII decorator that wraps an
underlying `Socket`. Implement mock symmetric record framing where
payloads are encrypted using an XOR-cipher with an authenticated HMAC
tag, demonstrating the layering of security boundaries over stream
descriptors.

``` cpp
class SecureSocket {
public:
    explicit SecureSocket(Socket raw_sock, std::array<uint8_t, 32> session_key);
    void send_secure(std::span<const std::byte> plaintext);
    std::vector<std::byte> recv_secure();
private:
    Socket m_sock;
    std::array<uint8_t, 32> m_key;
    uint64_t m_sequence_num{0};
    void encrypt_and_tag(std::span<const std::byte> in, std::vector<std::byte>& out);
    bool verify_and_decrypt(std::span<const std::byte> in, std::vector<std::byte>& out);
};
```

**Verification Tasks:**

1.  **Wire Ciphertext Verification:** Inspect the raw bytes emitted over
    the underlying network socket. Verify that none of the original
    plaintext characters appear in the raw transmission.
2.  **Tampering Detection:** Intercept a transmitted frame and flip a
    single byte in the ciphertext payload. Verify that `recv_secure()`
    detects the invalid authentication tag and rejects the frame.
3.  **Replay Attack Protection:** Duplicate a valid captured encrypted
    frame and replay it into the stream. Verify that the receiver uses
    sequence tracking to detect and reject the duplicated frame.

#### Exercise 17.12: Graceful Half-Close Teardown (`shutdown` Semantics)

**Context and Theory:**

When an application finishes writing data to a TCP connection, calling
POSIX `close()` terminates both read and write channels simultaneously.
If the remote peer is still transmitting data, the local kernel discards
incoming packets and sends an immediate `RST` (Reset) packet,
terminating the connection and causing data loss.

A clean communication protocol uses **TCP Half-Close** via the
`shutdown()` system call (as outlined in Figure 17–18):

``` cpp
shutdown(fd, SHUT_WR);
```

This transmits a `FIN` packet to the remote peer, signalling: *"I have
finished sending data, but I am still listening for your response."* The
remote peer reads until `recv()` returns $0$ (EOF), finishes its
computations, transmits the final response, and then closes its end.

<figure id="fig:close-teardown">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MTYuMiIgaGVp
Z2h0PSIxOTEuMjAwMDEiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNjE2LjIgMTkxLjIw
MDAxIj4KICA8c3R5bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xl
LCAuc3ZnYm9iIHJlY3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJv
a2Utd2lkdGg6IDEuNTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAg
c3Ryb2tlLWxpbmVjYXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdi
b2IgdGV4dCB7CiAgd2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWls
eTogQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6
IDEzcHg7Cn0KCi5zdmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6
IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7
Cn0KCi5zdmdib2IgLmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxl
ZCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5u
b2ZpbGwgewogIGZpbGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93
IHsKICBtYXJrZXItZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fy
cm93IHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9kaWFtb25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUp
Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwo
I29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBt
YXJrZXItc3RhcnQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Jp
Z19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVy
bCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9
ImFycm93IiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRo
PSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAg
PHBvbHlnb24gcG9pbnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2Vy
PgogICAgPG1hcmtlciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiBy
ZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+
PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjIiIGNsYXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJvcGVuX2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+
PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtl
cj4KICA8L2RlZnM+CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9
IjYxNi4yIiBoZWlnaHQ9IjE5MS4yMDAwMSI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjkuNzUiIHk9IjExLjcwMDAwMSIgPkNsaWVudDwvdGV4dD4KICA8bGluZSB4MT0i
MjcuMzAwMDAxIiB5MT0iMTUuNiIgeDI9IjI3LjMwMDAwMSIgeTI9IjE3MS42IiBjbGFzcz0ic29s
aWQiPjwvbGluZT4KICA8cGF0aCBkPSJNIDcwLjIwMDAwNSwwIEEgMTUuNiwxNS42IDAsMCwwIDcw
LjIwMDAwNSwxNS42IiBjbGFzcz0ibm9maWxsIj48L3BhdGg+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iNzIuMTUiIHk9IjExLjcwMDAwMSIgPlVwbG9hZHM8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTM0LjU1IiB5PSIxMS43MDAwMDEiID5GaWxlPC90ZXh0
PgogIDxwYXRoIGQ9Ik0gMTYzLjgsMCBBIDE1LjYsMTUuNiAwLDAsMSAxNjMuOCwxNS42IiBjbGFz
cz0ibm9maWxsIj48L3BhdGg+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDIzLjE1
MDAyIiB5PSIxMS43MDAwMDEiID5TZXJ2ZXI8L3RleHQ+CiAgPGxpbmUgeDE9IjQ0MC43IiB5MT0i
MTUuNiIgeDI9IjQ0MC43IiB5Mj0iMTQwLjQwMDAxIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8
bGluZSB4MT0iMzkiIHkxPSIzOSIgeDI9IjYyLjQiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xp
bmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzIuMTUiIHk9IjQyLjkiID5bRGF0
YTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMTguOTUwMDA1IiB5PSI0
Mi45IiA+Q2h1bmtzPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE3My41
NSIgeT0iNDIuOSIgPjE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTg5
LjE1MDAxIiB5PSI0Mi45IiA+Li48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMjEyLjU1IiB5PSI0Mi45IiA+Tl08L3RleHQ+CiAgPGxpbmUgeDE9IjIzNCIgeTE9IjM5IiB4
Mj0iNDIxLjIiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRz
PSI0MjEuMiwzNS4xMDAwMDIgNDI5LDM5IDQyMS4yLDQyLjkiIGNsYXNzPSJmaWxsZWQiPjwvcG9s
eWdvbj4KICA8bGluZSB4MT0iMzkiIHkxPSI3MC4yMDAwMDUiIHgyPSI2Mi40IiB5Mj0iNzAuMjAw
MDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8bGluZSB4MT0iMjE4LjQwMDAxIiB5MT0iNzAu
MjAwMDA1IiB4Mj0iMjQxLjgiIHkyPSI3MC4yMDAwMDUiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
IDxwb2x5Z29uIHBvaW50cz0iMjQxLjgsNjYuMyAyNDkuNiw3MC4yMDAwMDUgMjQxLjgsNzQuMSIg
Y2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjI1OS4zNSIgeT0iNzQuMSIgPltGSU48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iMjk4LjM1IiB5PSI3NC4xIiA+UGFja2V0XTwvdGV4dD4KICA8bGluZSB4MT0iMzU4Ljgw
MDAyIiB5MT0iNzAuMjAwMDA1IiB4Mj0iNDIxLjIiIHkyPSI3MC4yMDAwMDUiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iNDIxLjIsNjYuMyA0MjksNzAuMjAwMDA1IDQy
MS4yLDc0LjEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8cG9seWdvbiBwb2ludHM9IjQ2
LjgwMDAwMyw5Ny41IDM5LDEwMS40IDQ2LjgwMDAwMywxMDUuMyIgY2xhc3M9ImZpbGxlZCI+PC9w
b2x5Z29uPgogIDxsaW5lIHgxPSI0Ni44MDAwMDMiIHkxPSIxMDEuNCIgeDI9IjcwLjIwMDAwNSIg
eTI9IjEwMS40IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI3OS45NTAwMDUiIHk9IjEwNS4zIiA+W0ZpbmFsPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjEzNC41NSIgeT0iMTA1LjMiID5BZ2dyZWdhdGVkPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIyMC4zNSIgeT0iMTA1LjMiID5SZXBvcnRd
PC90ZXh0PgogIDxsaW5lIHgxPSIyODAuODAwMDIiIHkxPSIxMDEuNCIgeDI9IjQyOSIgeTI9IjEw
MS40IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjQ2LjgwMDAwMywx
MjguNyAzOSwxMzIuNiA0Ni44MDAwMDMsMTM2LjUiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4K
ICA8bGluZSB4MT0iNDYuODAwMDAzIiB5MT0iMTMyLjYiIHgyPSI3MC4yMDAwMDUiIHkyPSIxMzIu
NiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9IjE0OC4yIiB5MT0iMTMyLjYiIHgy
PSIxNzEuNiIgeTI9IjEzMi42IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjE3MS42LDEyOC43IDE3OS40MDAwMSwxMzIuNiAxNzEuNiwxMzYuNSIgY2xhc3M9ImZpbGxl
ZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE4OS4xNTAwMSIg
eT0iMTM2LjUiID5bRklOPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIy
OC4xNTAwMSIgeT0iMTM2LjUiID5QYWNrZXRdPC90ZXh0PgogIDxsaW5lIHgxPSIyODguNiIgeTE9
IjEzMi42IiB4Mj0iNDI5IiB5Mj0iMTMyLjYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcyLjE1IiB5PSI3NC4xIiA+c2h1dGRvd24oU0hVVF9X
Uik8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDU0LjM1IiB5PSI4OS43
MDAwMDUiID4oU2VydmVyIGRldGVjdHMgRU9GKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSI3OS45NTAwMDUiIHk9IjEzNi41IiA+Y2xvc2UoKTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MC45NSIgeT0iMTY3LjciID4oQ2xpZW50IHJlYWRzIEVP
RiwgdGhlbiBjbG9zZXMpPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 17–18: Half-Close Teardown</figcaption>
</figure>

**Specification:**

Implement a client-server file-upload protocol using half-close
semantics. The client streams an arbitrary payload, invokes
`shutdown(SHUT_WR)`, reads the server's final calculation report, and
then closes the socket completely.

``` cpp
class HalfCloseUploader {
public:
    static void upload_and_await_receipt(Socket& sock, std::span<const std::byte> data) {
        sock.send_all(data);
        if (::shutdown(sock.native_handle(), SHUT_WR) < 0) {
            throw std::runtime_error("Shutdown failed: " + std::string(strerror(errno)));
        }
        std::vector<std::byte> response;
        std::array<std::byte, 512> buf;
        while (true) {
            ssize_t bytes = ::recv(sock.native_handle(), buf.data(), buf.size(), 0);
            if (bytes <= 0) break; // Clean EOF
            response.insert(response.end(), buf.begin(), buf.begin() + bytes);
        }
    }
};
```

**Verification Tasks:**

1.  **Clean EOF Detection:** Run a server receiving an upload. Verify
    that the server loop terminates its read phase when `recv()` returns
    $0$, without raising socket errors or timeouts.
2.  **Bi-directional Asymmetry:** Verify that after invoking
    `shutdown(SHUT_WR)`, subsequent calls to `recv()` on the client
    succeed and capture the server's response.
3.  **Error on Write Post-Shutdown:** Attempt to call `send()` on a
    socket following a successful `shutdown(SHUT_WR)`. Verify that the
    operation fails and sets `errno = EPIPE` (Broken Pipe).